# Inclusões em pauta

Constrói e analisa a tabela de **inclusões em pauta** do STF entre 2020
e 2025, respondendo à pergunta:

> Qual é o total geral e o perfil das inclusões em pauta?

A análise permite segmentar por: ano, classe, tipo de questão (PR/RC/IJ),
relator, desfecho e ambiente (virtual ou físico) e correlacionar
essas variáveis entre si.

## Unidade de análise

A unidade de análise é **cada inserção de um processo na pauta**, não o
processo em si. Um mesmo processo pode aparecer várias vezes na tabela
(uma linha para cada vez que foi pautado).

Isso é intencional: permite capturar pedidos de vista, destaques e reinclusões como eventos distintos.

## Fontes dos dados

| Arquivo | Origem | Uso |
|---|---|---|
| `dim_andamentos.parquet` | Pré-processamento | Detectar eventos de inclusão em pauta |
| `dim_decisoes.parquet` | Pré-processamento | Identificar o desfecho de cada pauta |
| `arquivosConcatenados.parquet` | Pré-processamento | Trazer classe e relator |

## Critérios metodológicos

### Andamentos que configuram uma inclusão em pauta
Os seguintes valores de `and_nome` marcam uma inclusão em pauta:
- `Inclua-se em pauta - minuta extraída`
- `Incluído no calendário de julgamento pelo Presidente`
- `Incluído no calendário de julgamento pela Presidente`
- `Incluído no calendário de julgamento`
- `Incluído na lista de julgamento`
- `Apresentado em mesa para julgamento`
- `PROCESSO A JULGAMENTO - PAUTA, DJ:` (formato histórico — anos 1990)
- `PROCESSO EM MESA` (formato histórico)

### Ambiente (Virtual ou Físico)
Determinado pelo `and_complemento` do próprio evento de inclusão:
- Contém `"Julgamento Virtual"` → **Plenário Virtual**
- Não contém → **Plenário Físico**



### Tipo de questão (PR / RC / IJ)
Extraído do sufixo de classe no `and_complemento`, conforme
**Tabela 2 da dissertação de referência (p. 62)**:
- Sem sufixo → **PR** (julgamento de mérito)
- Sufixo com ED ou AgR (e combinações) → **RC** (recurso)
- Sufixo com MC, TPI, Ref, QO → **IJ** (questão incidental)



### Desfecho
Identificado no texto de `dec_complemento` em `dim_decisoes`,
para decisões onde `dec_julgador` contém "SESSÃO VIRTUAL":
1. **Não concluído - destaque** → `and_nome = "Processo destacado no Julgamento Virtual"` dentro de 30 dias após a inclusão (fonte: dim_andamentos)
2. **Não concluído - pedido de vista** → andamento de vista de ministro dentro de 30 dias (fonte: dim_andamentos)
3. **Concluído - decisão unânime** → texto de `dec_complemento` contém "unanimidade" (fonte: dim_decisoes)
4. **Concluído - decisão maioria, vencido o relator** → texto contém "maioria" + relator entre os vencidos
5. **Concluído - decisão maioria com o relator** → texto contém "maioria" sem relator vencido
6. **Não concluído - motivos diversos** → fallback quando nenhuma das condições acima é satisfeita

## 0. Importações

In [82]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns

from plotly.subplots import make_subplots

import re
import unicodedata

from google.colab import drive
from pathlib import Path

In [83]:
pip install pyarrow

In [84]:
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Plenário Virtual/data/processed')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [85]:
ANO_INI, ANO_FIM = 2020, 2025

In [86]:
ANO_INI, ANO_FIM = 2016, 2025
print(f"Novo período de análise definido: {ANO_INI} a {ANO_FIM}")

Novo período de análise definido: 2016 a 2025


## 1. Carregamento dos dados

Carregamos os três arquivos necessários. O `dim_andamentos` é carregado
em sua versão **original** (não categorizado), pois trabalharemos
diretamente com os valores de `and_nome` e `and_complemento`.

### Importando os datasets

In [87]:
df_and = pd.read_parquet(
    BASE / 'dim_andamentos.parquet',
    columns=['incidente', 'and_index', 'and_data',
             'and_nome', 'and_complemento']
)

In [88]:
df_dec = pd.read_parquet(
    BASE / 'dim_decisoes.parquet',
    columns=['incidente', 'dec_index', 'dec_data',
             'dec_complemento', 'dec_julgador']
)

In [89]:
df_fato = pd.read_parquet(
    BASE / 'arquivosConcatenados.parquet',
    columns=['incidente', 'classe', 'nome_processo', 'relator']
)

In [90]:
df_fato_completa = pd.read_parquet(
    BASE / 'arquivosConcatenados.parquet',
)

## 2. Funções auxiliares

### Variáveis e filtros

In [ ]:
from src.inclusao_pauta import (
    ANDAMENTOS_PAUTA, ANDAMENTOS_RETIRADA, DESTAQUE_NOMES, SESSION_VISTA_NAMES,
    eh_vista_ministro, classificar_desfecho_texto, extrair_janela_sessao,
)

### Extração do sufixo de classe



O `and_complemento` dos eventos virtuais aparece em dois formatos:

- **Formato moderno** (a partir de ~2020):
  - `"Julgamento Virtual: ADC-ED. Incluído na Lista 92-2024..."`

- **Formato antigo** (até ~2019):
  - `"Julgamento Virtual - Pleno em 04/09/2018 14:44:32 - ADC-AgR-ED"`

- **Sem sufixo** (julgamento de mérito):
  - `"Julgamento Virtual: . Incluído na Lista..."` → PR
  - `"Julgamento Virtual: Incluído na Lista..."` → PR

Recebe o texto do and_complemento de um evento de inclusão em pauta virtual e tenta extrair a classe processual com seu sufixo — que é a informação que diz o que está sendo julgado naquela sessão (ex: "ADI-ED", "ADI-MC-Ref", "ADI").

### Classificação do tipo de questão


Ela recebe um `sufixo` — que é o resultado da função `extrair_sufixo`, ou seja, o nome do processo com sua classe e sufixo extraídos do complemento. Por exemplo: `"ADI-ED"`, `"ADI-MC-Ref"`, `"ADI"`, ou `None`.

1. Se o sufixo for `None` → "Não identificado"
    ```python
    if sufixo is None:
        return 'Não identificado'
    ```
    - Isso acontece quando o `and_complemento` não tinha informação de classe — geralmente registros mais antigos.


2. Remove a classe processual e pega só os sufixos
    ```python
    partes = sufixo.upper().split('-')[1:]
    ```
    - Exemplo: `"ADI-MC-Ref"` vira `['MC', 'REF']`. O `[1:]` descarta o primeiro elemento (`"ADI"`), que é a classe e não interessa para a classificação.

3. Se não sobrou nada após remover a classe → PR
    ```python
    if not partes:
        return 'PR'
    ```
    - Isso acontece quando o sufixo era só a classe, sem nenhum complemento — exemplo: `"ADI"` vira `[]` após o split. Sem sufixo = julgamento de mérito = **PR (Principal)**.

4. Junta as partes num texto para aplicar as regex
    ```python
    s = '-'.join(partes)
    ```
    - Exemplo: `['MC', 'REF']` vira `"MC-REF"`. Isso facilita a busca por padrões.

5. Verifica se é RC (Recurso)
    ```python
    if re.search(r'\bED\b|AGR', s):
        return 'RC'
    ```
    - Busca `ED` (embargos de declaração) ou `AGR` (agravo regimental) no sufixo. O `\b` em volta de `ED` garante que ele só case como palavra inteira — evitando falsos positivos se `ED` aparecesse como parte de outra palavra. `AGR` não precisa de `\b` porque não há palavras que comecem com essas 3 letras.

    - Exemplos que viram RC: `"ADI-ED"`, `"ADI-AgR"`, `"ADI-MC-Ref-ED"` (tem ED no final), `"ADI-ED-ED-segundos"`.


6. Verifica se é IJ (Questão Incidental)
    ```python
    if re.search(r'\bMC\b|\bTPI\b|\bREF\b|\bQO\b|ACORDO', s):
        return 'IJ'
    ```
    - Busca MC (medida cautelar), TPI (tutela provisória incidental), REF (referendo), QO (questão de ordem) ou ACORDO.
    - Exemplos que viram IJ: `"ADI-MC"`, `"ADI-MC-Ref"`, `"ADI-TPI-Ref"`.


7. Se nada casou → "Não identificado"
    ```python
    return 'Não identificado'
    ```
    - Para sufixos que existem mas não encaixam em nenhuma categoria conhecida.

> A verificação de RC vem **antes** de IJ deliberadamente. Isso cobre casos como `"ADI-MC-Ref-ED"` — que tem tanto `MC-Ref` (que seria IJ) quanto `ED` (que é RC). Nesses casos, o que está sendo julgado naquela sessão é o **recurso mais externo** (o `ED`), não mais a medida cautelar original. Por isso RC tem prioridade.


### Normalização do texto


Recebe qualquer texto e devolve uma versão simplificada: sem acentos, tudo em minúsculo. O objetivo é facilitar buscas por palavras-chave sem se preocupar com variações de grafia.



### Pedido de vista


Recebe o nome e o complemento de um andamento e decide se aquele andamento representa um pedido de vista de ministro durante uma sessão virtual. Retorna True se for, False se não for.

### Classificação do desfecho


Recebe o texto livre do campo dec_complemento de uma decisão do Plenário Virtual e tenta identificar qual foi o resultado do julgamento. Retorna uma string com o desfecho classificado, ou None se o texto não contiver informação suficiente.

É responsável apenas pelos desfechos concluídos — vista e destaque são identificados antes desta função, via dim_andamentos.

### Extração da janela

Recebe uma tabela de eventos (de destaque ou de vista), um número de incidente e uma data de pauta, e responde com True ou False uma pergunta simples: esse processo teve algum evento desse tipo nos 30 dias após essa pauta?
É usada duas vezes no loop de classificação de desfecho — uma para destaque e uma para vista de ministro.

## 3. Seleção e classificação dos eventos de inclusão em pauta



Para cada evento de inclusão:
1. Verificamos `and_complemento`: contém "Julgamento Virtual" → Virtual; senão → Físico
2. Extraímos o sufixo de classe do complemento (só para virtuais)
3. Classificamos o tipo de questão (PR/RC/IJ) pelo sufixo

In [ ]:
from src.inclusao_pauta import selecionar_e_classificar_pauta

df_virt_pauta, df_fis_pauta = selecionar_e_classificar_pauta(df_and)
df_pauta = pd.concat([df_virt_pauta, df_fis_pauta])

print(f"Eventos de inclusão em pauta: {len(df_pauta):,}")
print(f"  Plenário Virtual: {len(df_virt_pauta):,}")
print(f"  Plenário Físico:  {len(df_fis_pauta):,}")

print(f"\nTipo de questão (virtual):")
print(df_virt_pauta['tipo_questao'].value_counts().to_string())

## 4. Identificação do desfecho


In [ ]:
from src.inclusao_pauta import limpar_decisoes_virtuais, classificar_desfecho_virtual

dec_virt = limpar_decisoes_virtuais(df_dec)
print(f"Decisões virtuais (após limpeza): {len(dec_virt):,}")

df_virt_pauta, df_and_virt = classificar_desfecho_virtual(df_virt_pauta, df_and, dec_virt)

print("\nDesfecho (Plenário Virtual):")
print(df_virt_pauta['desfecho'].value_counts().to_string())

In [ ]:
from src.inclusao_pauta import classificar_desfecho_fisico

df_fis_pauta = classificar_desfecho_fisico(df_fis_pauta, df_and, df_dec)

print("\nDesfecho (Plenário Físico):")
print(df_fis_pauta['desfecho'].value_counts().to_string())

In [ ]:
from src.inclusao_pauta import montar_sessoes_virtuais

df_sessoes = montar_sessoes_virtuais(df_and, df_virt_pauta, df_and_virt, dec_virt, ano_ini=ANO_INI, ano_fim=ANO_FIM)

incl_periodo = df_virt_pauta[df_virt_pauta['and_data_dt'].dt.year.between(ANO_INI, ANO_FIM)]

print(f"Sessões virtuais iniciadas ({ANO_INI}-{ANO_FIM}): {len(df_sessoes):,}")
print(f"Processos distintos: {df_sessoes['incidente'].nunique():,}")

print(f"\nInclusões virtuais no período: {len(incl_periodo):,}")
print(f"  Viraram sessão:     {incl_periodo['virou_sessao'].sum():,} "
      f"({100*incl_periodo['virou_sessao'].mean():.1f}%)")
print(f"  NÃO viraram sessão: {(~incl_periodo['virou_sessao']).sum():,} "
      f"({100*(~incl_periodo['virou_sessao']).mean():.1f}%)")

print(f"\nTipo de questão (sessões virtuais):")
print(df_sessoes['tipo_questao'].value_counts().to_string())

print("\nDesfecho das sessões virtuais iniciadas:")
print(df_sessoes['desfecho'].value_counts().to_string())
print("\nMacro-desfecho:")
print(df_sessoes['macro_desfecho'].value_counts().to_string())

## Montagem da tabela final


Une pautas virtuais e físicas, enriquece com classe e relator,
filtra para 2020–2025 e padroniza as colunas.

In [ ]:
from src.inclusao_pauta import montar_tabela_final, montar_tabela_sessoes

df_final = montar_tabela_final(df_virt_pauta, df_fis_pauta, df_fato, ano_ini=ANO_INI, ano_fim=ANO_FIM)
df_sessoes_final = montar_tabela_sessoes(df_sessoes, df_fato)

### 5.1 Salvamento do Período Ampliado (2016-2025)
Executamos a filtragem e salvamento final para o novo intervalo solicitado pelo usuário.

In [105]:
# ── Salvamento (parquet + csv, sem reprocessamento duplicado) ────────────
df_final.to_parquet(BASE / 'inclusoes_em_pauta.parquet', index=False)
df_sessoes_final.to_parquet(BASE / 'sessoes_virtuais.parquet', index=False)

#df_final.to_csv(BASE / 'inclusoes_em_pauta_2016_2025.csv', index=False, sep=';', encoding='utf-8-sig')
#df_sessoes_final.to_csv(BASE / 'sessoes_virtuais_2016_2025.csv', index=False, sep=';', encoding='utf-8-sig')

print(f"Processamento concluído: {len(df_final):,} inclusões e {len(df_sessoes_final):,} sessões ({ANO_INI}-{ANO_FIM}).")
display(df_final.head())

Processamento concluído: 10,285 inclusões e 4,807 sessões (2016-2025).


,incidente,nome_processo,classe,relator,ano,data_inclusao,data_inclusao_dt,ambiente,tipo_questao,tipo_questao_original,sufixo_extraido,desfecho,macro_desfecho,andamento_origem,virou_sessao
0,11234,ADI 3423,ADI,GILMAR MENDES,2020,12/05/2020,2020-05-12,Plenário Virtual,PR,Não identificado,None,Concluído - decisão maioria com o relator,Concluído,Inclua-se em pauta - minuta extraída,True
1,11299,ADI 2135,ADI,CÁRMEN LÚCIA,2018,24/04/2018,2018-04-24,Plenário Virtual,RC,RC,ADI-AgR,Concluído - decisão maioria com o relator,Concluído,Inclua-se em pauta - minuta extraída,True
2,11299,ADI 2135,ADI,CÁRMEN LÚCIA,2025,23/06/2025,2025-06-23,Plenário Virtual,PR,Não identificado,None,Concluído - decisão unânime,Concluído,Inclua-se em pauta - minuta extraída,True
3,11299,ADI 2135,ADI,CÁRMEN LÚCIA,2025,23/06/2025,2025-06-23,Plenário Virtual,PR,Não identificado,None,Concluído - decisão unânime,Concluído,Inclua-se em pauta - minuta extraída,True
4,12151,ADPF 183,ADPF,ALEXANDRE DE MORAES,2019,10/09/2019,2019-09-10,Plenário Virtual,PR,Não identificado,None,Concluído - decisão unânime,Concluído,Inclua-se em pauta - minuta extraída,True


## Diagnóstico

In [106]:
def resumo_geral(df, titulo, unidade='inclusões'):
    """Panorama de um dataset: volume, ambiente, ano e classe."""
    print(f"\n{'='*62}")
    print(f"{titulo.upper()}")
    print(f"{'='*62}")
    print(f"Total de {unidade}: {len(df):,}")
    print(f"Processos distintos: {df['incidente'].nunique():,}")
    print(f"Média de {unidade} por processo: {len(df)/df['incidente'].nunique():.2f}")

    if df['ambiente'].nunique() > 1:
        print(f"\nPor ambiente:")
        for amb, n in df['ambiente'].value_counts().items():
            print(f"  {amb:<20} {n:>6,} ({100*n/len(df):>5.1f}%)")

    print(f"\nPor ano:")
    por_ano = df['ano'].value_counts().sort_index()
    for ano, n in por_ano.items():
        barra = '█' * int(30 * n / por_ano.max())
        print(f"  {ano}  {n:>5,}  {barra}")

    print(f"\nPor classe:")
    for cl, n in df['classe'].value_counts().items():
        print(f"  {cl:<6} {n:>6,} ({100*n/len(df):>5.1f}%)")


resumo_geral(df_final, 'Inclusões em pauta (2020–2025)', 'inclusões')
resumo_geral(df_sessoes_final, 'Sessões virtuais iniciadas (2020–2025)', 'sessões')


INCLUSÕES EM PAUTA (2020–2025)
Total de inclusões: 10,285
Processos distintos: 3,644
Média de inclusões por processo: 2.82

Por ambiente:
  Plenário Virtual      5,422 ( 52.7%)
  Plenário Físico       4,863 ( 47.3%)

Por ano:
  2016    299  █████
  2017    405  ███████
  2018    842  ███████████████
  2019  1,222  ██████████████████████
  2020  1,595  ██████████████████████████████
  2021  1,298  ████████████████████████
  2022  1,182  ██████████████████████
  2023  1,388  ██████████████████████████
  2024  1,081  ████████████████████
  2025    973  ██████████████████

Por classe:
  ADI     8,324 ( 80.9%)
  ADPF    1,582 ( 15.4%)
  ADC       217 (  2.1%)
  ADO       162 (  1.6%)

SESSÕES VIRTUAIS INICIADAS (2020–2025)
Total de sessões: 4,807
Processos distintos: 2,968
Média de sessões por processo: 1.62

Por ano:
  2016     11  
  2017     38  █
  2018     74  ██
  2019    349  ████████████
  2020    871  ██████████████████████████████
  2021    793  ███████████████████████████
  2022

In [107]:
# ===========================================================================
# 6.2 — Diagnóstico por ambiente e unidade
# ===========================================================================
def diagnostico(df, titulo, unidade='inclusões'):
    """
    Tipo de questão, desfecho e evolução anual.
    Percentuais em todas as distribuições — números absolutos sozinhos
    dificultam a comparação entre ambientes de tamanhos diferentes.
    """
    print(f"\n{'='*62}")
    print(f"{titulo.upper()}")
    print(f"{'='*62}")
    print(f"Total de {unidade}: {len(df):,}  |  "
          f"Processos: {df['incidente'].nunique():,}")

    print(f"\nTipo de questão:")
    for tq, n in df['tipo_questao'].value_counts().items():
        print(f"  {tq:<20} {n:>6,} ({100*n/len(df):>5.1f}%)")

    print(f"\nDesfecho:")
    for d, n in df['desfecho'].value_counts().items():
        print(f"  {d:<48} {n:>5,} ({100*n/len(df):>5.1f}%)")

    print(f"\nMacro-desfecho:")
    macro = df['macro_desfecho'].value_counts()
    for m, n in macro.items():
        print(f"  {m:<20} {n:>6,} ({100*n/len(df):>5.1f}%)")
    print(f"  {'→ Taxa de conclusão':<20} "
          f"{100*macro.get('Concluído', 0)/len(df):>11.1f}%")

    print(f"\nTaxa de conclusão por ano:")
    for ano in sorted(df['ano'].unique()):
        sub = df[df['ano'] == ano]
        taxa = 100 * (sub['macro_desfecho'] == 'Concluído').mean()
        barra = '█' * int(taxa / 3)
        print(f"  {ano}  {taxa:>5.1f}%  {barra}")

    print(f"\nDesfecho detalhado por ano:")
    print(df.groupby(['ano', 'desfecho']).size().unstack(fill_value=0).to_string())


diagnostico(
    df_final[df_final['ambiente'] == 'Plenário Virtual'],
    'Inclusões em pauta — Plenário Virtual', 'inclusões'
)
diagnostico(
    df_final[df_final['ambiente'] == 'Plenário Físico'],
    'Inclusões em pauta — Plenário Físico', 'inclusões'
)
diagnostico(
    df_sessoes_final,
    'Sessões virtuais iniciadas', 'sessões'
)


INCLUSÕES EM PAUTA — PLENÁRIO VIRTUAL
Total de inclusões: 5,422  |  Processos: 3,047

Tipo de questão:
  PR                    3,805 ( 70.2%)
  RC                    1,237 ( 22.8%)
  IJ                      380 (  7.0%)

Desfecho:
  Concluído - decisão unânime                      2,340 ( 43.2%)
  Não concluído - pedido de vista                    917 ( 16.9%)
  Concluído - decisão maioria com o relator          846 ( 15.6%)
  Não concluído - retirado de pauta                  782 ( 14.4%)
  Não concluído - motivos diversos                   225 (  4.1%)
  Não concluído - destaque                           172 (  3.2%)
  Concluído - decisão maioria, vencido o relator     140 (  2.6%)

Macro-desfecho:
  Concluído             3,326 ( 61.3%)
  Não concluído         2,096 ( 38.7%)
  → Taxa de conclusão         61.3%

Taxa de conclusão por ano:
  2016   75.0%  █████████████████████████
  2017   50.0%  ████████████████
  2018   70.2%  ███████████████████████
  2019   68.5%  ████████████████

In [108]:
# ===========================================================================
# 6.3 — Inclusão em pauta × sessão iniciada
# ===========================================================================
# Usa a coluna 'virou_sessao' JÁ CALCULADA na Célula 4c (via janela agendada
# do complemento). Não recalcular aqui com tolerância fixa de dias: pautas de
# dezembro agendadas para fevereiro atravessam o recesso (~52 dias) e seriam
# cortadas indevidamente, produzindo números divergentes da 4c.

df_final_virt = df_final[df_final['ambiente'] == 'Plenário Virtual']

print(f"\n{'='*62}")
print(f"INCLUSÃO EM PAUTA × SESSÃO INICIADA — Plenário Virtual")
print(f"{'='*62}")

n_virou = df_final_virt['virou_sessao'].sum()
n_nao   = (~df_final_virt['virou_sessao']).sum()
print(f"Inclusões virtuais: {len(df_final_virt):,}")
print(f"  Viraram sessão:     {n_virou:>5,} ({100*n_virou/len(df_final_virt):>5.1f}%)")
print(f"  NÃO viraram sessão: {n_nao:>5,} ({100*n_nao/len(df_final_virt):>5.1f}%)")

print(f"\nDesfecho das que NÃO viraram sessão:")
nao_virou = df_final_virt[~df_final_virt['virou_sessao']]
for d, n in nao_virou['desfecho'].value_counts().items():
    print(f"  {d:<48} {n:>4,} ({100*n/len(nao_virou):>5.1f}%)")
print(f"\n  A maioria é retirada de pauta — o processo foi pautado mas")
print(f"  retirado antes de a sessão começar. É o caso que valida a")
print(f"  distinção entre as duas unidades de análise.")
print(f"\n  Nota: inclusões de dez/2025 agendadas para fev/2026 aparecem")
print(f"  como 'não viraram sessão' por efeito de borda do recorte.")

# Taxa de conversão por ano
print(f"\nTaxa de conversão pauta → sessão, por ano:")
for ano in sorted(df_final_virt['ano'].unique()):
    sub = df_final_virt[df_final_virt['ano'] == ano]
    taxa = 100 * sub['virou_sessao'].mean()
    barra = '█' * int(taxa / 3)
    print(f"  {ano}  {taxa:>5.1f}%  {barra}  ({sub['virou_sessao'].sum():,}/{len(sub):,})")

# Comparação lado a lado das duas unidades
print(f"\n{'='*62}")
print(f"AS DUAS UNIDADES LADO A LADO")
print(f"{'='*62}")
print(f"{'Métrica':<42} {'Inclusões':>9} {'Sessões':>9}")
print(f"{'-'*62}")
print(f"{'Total de eventos':<42} {len(df_final_virt):>9,} {len(df_sessoes_final):>9,}")
print(f"{'Processos distintos':<42} "
      f"{df_final_virt['incidente'].nunique():>9,} "
      f"{df_sessoes_final['incidente'].nunique():>9,}")

taxa_i = 100 * (df_final_virt['macro_desfecho'] == 'Concluído').mean()
taxa_s = 100 * (df_sessoes_final['macro_desfecho'] == 'Concluído').mean()
print(f"{'Taxa de conclusão':<42} {taxa_i:>8.1f}% {taxa_s:>8.1f}%")

print(f"{'-'*62}")
for d in sorted(set(df_final_virt['desfecho']) | set(df_sessoes_final['desfecho'])):
    ni = (df_final_virt['desfecho'] == d).sum()
    ns = (df_sessoes_final['desfecho'] == d).sum()
    print(f"{d:<42} {ni:>9,} {ns:>9,}")

print(f"\nA análise por SESSÃO é mais limpa: exclui as pautas que nunca")
print(f"chegaram a julgamento. 'Motivos diversos' cai de {100*(df_final_virt['desfecho']=='Não concluído - motivos diversos').mean():.1f}% "
      f"para {100*(df_sessoes_final['desfecho']=='Não concluído - motivos diversos').mean():.1f}%.")


INCLUSÃO EM PAUTA × SESSÃO INICIADA — Plenário Virtual
Inclusões virtuais: 5,422
  Viraram sessão:     4,973 ( 91.7%)
  NÃO viraram sessão:   449 (  8.3%)

Desfecho das que NÃO viraram sessão:
  Não concluído - retirado de pauta                 391 ( 87.1%)
  Não concluído - motivos diversos                   41 (  9.1%)
  Concluído - decisão unânime                         8 (  1.8%)
  Não concluído - destaque                            4 (  0.9%)
  Não concluído - pedido de vista                     3 (  0.7%)
  Concluído - decisão maioria com o relator           2 (  0.4%)

  A maioria é retirada de pauta — o processo foi pautado mas
  retirado antes de a sessão começar. É o caso que valida a
  distinção entre as duas unidades de análise.

  Nota: inclusões de dez/2025 agendadas para fev/2026 aparecem
  como 'não viraram sessão' por efeito de borda do recorte.

Taxa de conversão pauta → sessão, por ano:
  2016   91.7%  ██████████████████████████████  (11/12)
  2017   78.3%  ███████

In [109]:
# ===========================================================================
# 6.4 — Por evento vs por processo
# ===========================================================================
# Duas leituras da mesma realidade:
#   - por EVENTO: cada inclusão/sessão conta. Mede o esforço do tribunal.
#   - por PROCESSO: cada processo conta uma vez. Mede o resultado alcançado.
# A diferença revela quantas vezes o STF pauta antes de julgar.

def desfecho_final_processo(grupo, col_data):
    """
    Desfecho final de um processo, a partir de todos os seus eventos.
      - Se houve conclusão, retorna a ÚLTIMA conclusão
      - Se nunca concluiu, retorna o último evento
    """
    g = grupo.sort_values(col_data)
    concluidas = g[g['macro_desfecho'] == 'Concluído']
    if not concluidas.empty:
        return concluidas.iloc[-1]['desfecho']
    return g.iloc[-1]['desfecho']


def comparar_evento_vs_processo(df, titulo, col_data, unidade='pautas'):
    """Compara o desfecho por evento com o desfecho final por processo."""
    desf_proc = (
        df.groupby('incidente')
        .apply(lambda g: desfecho_final_processo(g, col_data), include_groups=False)
        .reset_index(name='desfecho_final_processo')
    )

    n_ev, n_pr = len(df), len(desf_proc)

    print(f"\n{'='*62}")
    print(f"{titulo.upper()}")
    print(f"POR {unidade.upper()} vs POR PROCESSO")
    print(f"{'='*62}")
    print(f"{unidade.capitalize()}: {n_ev:,}  |  Processos: {n_pr:,}  |  "
          f"Média: {n_ev/n_pr:.2f} {unidade}/processo")

    # Tabela comparativa
    vc_ev = df['desfecho'].value_counts()
    vc_pr = desf_proc['desfecho_final_processo'].value_counts()

    print(f"\n{'Desfecho':<48} {'Por ' + unidade:>12} {'Por processo':>13}")
    print(f"{'-'*75}")
    for d in sorted(set(vc_ev.index) | set(vc_pr.index)):
        e, p = vc_ev.get(d, 0), vc_pr.get(d, 0)
        print(f"{d:<48} {e:>6,} ({100*e/n_ev:>4.1f}%) {p:>5,} ({100*p/n_pr:>4.1f}%)")

    # Taxas de conclusão
    taxa_ev = 100 * (df['macro_desfecho'] == 'Concluído').mean()
    taxa_pr = 100 * desf_proc['desfecho_final_processo'].str.startswith('Concluído').mean()

    print(f"\n{'-'*75}")
    print(f"{'TAXA DE CONCLUSÃO':<48} {taxa_ev:>11.1f}% {taxa_pr:>12.1f}%")
    print(f"\n  Diferença de {taxa_pr - taxa_ev:+.1f} pontos: o tribunal pauta em média")
    print(f"  {n_ev/n_pr:.2f} vezes cada processo até julgá-lo.")

    return desf_proc


desf_proc_virtual = comparar_evento_vs_processo(
    df_final[df_final['ambiente'] == 'Plenário Virtual'],
    'Inclusões em pauta — Plenário Virtual',
    col_data='data_inclusao_dt', unidade='pautas'
)
desf_proc_fisico = comparar_evento_vs_processo(
    df_final[df_final['ambiente'] == 'Plenário Físico'],
    'Inclusões em pauta — Plenário Físico',
    col_data='data_inclusao_dt', unidade='pautas'
)
desf_proc_sessoes = comparar_evento_vs_processo(
    df_sessoes_final,
    'Sessões virtuais iniciadas',
    col_data='data_sessao_dt', unidade='sessões'
)


INCLUSÕES EM PAUTA — PLENÁRIO VIRTUAL
POR PAUTAS vs POR PROCESSO
Pautas: 5,422  |  Processos: 3,047  |  Média: 1.78 pautas/processo

Desfecho                                           Por pautas  Por processo
---------------------------------------------------------------------------
Concluído - decisão maioria com o relator           846 (15.6%)   622 (20.4%)
Concluído - decisão maioria, vencido o relator      140 ( 2.6%)   102 ( 3.3%)
Concluído - decisão unânime                       2,340 (43.2%) 1,911 (62.7%)
Não concluído - destaque                            172 ( 3.2%)    83 ( 2.7%)
Não concluído - motivos diversos                    225 ( 4.1%)    90 ( 3.0%)
Não concluído - pedido de vista                     917 (16.9%)    89 ( 2.9%)
Não concluído - retirado de pauta                   782 (14.4%)   150 ( 4.9%)

---------------------------------------------------------------------------
TAXA DE CONCLUSÃO                                       61.3%         86.5%

  Diferença de

In [110]:
# ===========================================================================
# 6.5 — Verificações de integridade
# ===========================================================================
# Confere invariantes que devem valer sempre. Qualquer alerta indica
# que algo mudou no pipeline e merece investigação.

print(f"\n{'='*62}")
print(f"VERIFICAÇÕES DE INTEGRIDADE")
print(f"{'='*62}")

checks = []

# Recorte temporal
fora = ~df_final['ano'].between(ANO_INI, ANO_FIM)
checks.append(('Todas as inclusões em 2020–2025', fora.sum() == 0,
               f"{fora.sum()} fora do recorte"))

fora_s = ~df_sessoes_final['ano'].between(ANO_INI, ANO_FIM)
checks.append(('Todas as sessões em 2020–2025', fora_s.sum() == 0,
               f"{fora_s.sum()} fora do recorte"))

# Ausências
checks.append(('Sem desfecho nulo', df_final['desfecho'].isna().sum() == 0,
               f"{df_final['desfecho'].isna().sum()} nulos"))
checks.append(('Sem classe nula', df_final['classe'].isna().sum() == 0,
               f"{df_final['classe'].isna().sum()} nulos"))

# Coerência desfecho ↔ macro_desfecho
inc = df_final[
    (df_final['desfecho'].str.startswith('Concluído')) &
    (df_final['macro_desfecho'] != 'Concluído')
]
checks.append(('desfecho e macro_desfecho coerentes', len(inc) == 0,
               f"{len(inc)} incoerentes"))

# Conversão de tipo aplicada só ao Virtual
ni_virt = ((df_final['ambiente'] == 'Plenário Virtual') &
           (df_final['tipo_questao'] == 'Não identificado')).sum()
checks.append(('Virtual sem "Não identificado"', ni_virt == 0,
               f"{ni_virt} casos"))

ni_fis = ((df_final['ambiente'] == 'Plenário Físico') &
          (df_final['tipo_questao'] == 'Não identificado')).sum()
checks.append(('Físico preserva "Não identificado"', ni_fis > 0,
               f"{ni_fis} casos (esperado — complemento sem sufixo)"))

# Classes do controle concentrado
classes_ok = set(df_final['classe'].dropna()) <= {'ADI', 'ADPF', 'ADC', 'ADO'}
checks.append(('Só classes do controle concentrado', classes_ok,
               f"{set(df_final['classe'].dropna())}"))

# Sessões ⊆ processos virtuais
inc_virt = set(df_final[df_final['ambiente'] == 'Plenário Virtual']['incidente'])
orfas = set(df_sessoes_final['incidente']) - inc_virt
checks.append(('Sessões pertencem a processos pautados', len(orfas) == 0,
               f"{len(orfas)} processos com sessão sem inclusão no recorte"))

for nome, ok, detalhe in checks:
    print(f"  [{'OK ' if ok else 'ALERTA'}] {nome:<42} {detalhe}")

n_alertas = sum(1 for _, ok, _ in checks if not ok)
if n_alertas == 0:
    print(f"\n  Todas as verificações passaram.")
else:
    print(f"\n  {n_alertas} alerta(s) — investigar antes de usar os resultados.")
    print(f"  Nota: 'Sessões pertencem a processos pautados' pode alertar")
    print(f"  legitimamente se houver processo pautado em 2019 com sessão em 2020.")


VERIFICAÇÕES DE INTEGRIDADE
  [OK ] Todas as inclusões em 2020–2025            0 fora do recorte
  [OK ] Todas as sessões em 2020–2025              0 fora do recorte
  [OK ] Sem desfecho nulo                          0 nulos
  [OK ] Sem classe nula                            0 nulos
  [OK ] desfecho e macro_desfecho coerentes        0 incoerentes
  [OK ] Virtual sem "Não identificado"             0 casos
  [OK ] Físico preserva "Não identificado"         4306 casos (esperado — complemento sem sufixo)
  [OK ] Só classes do controle concentrado         {'ADI', 'ADPF', 'ADC', 'ADO'}
  [ALERTA] Sessões pertencem a processos pautados     4 processos com sessão sem inclusão no recorte

  1 alerta(s) — investigar antes de usar os resultados.
  Nota: 'Sessões pertencem a processos pautados' pode alertar
  legitimamente se houver processo pautado em 2019 com sessão em 2020.


## Extras

### Refinando os motivos diversos

O que a refinação faz:
Cria a coluna desfecho_refinado — mantém todos os desfechos originais intactos, exceto os "motivos diversos" do PP, que são desdobrados em três subcategorias:

"sem decisão registrada" (78,4%) — a pauta não gerou nenhuma decisão colegiada. É o caso mais limpo: pautou, não julgou.
"registro sem fórmula de votação" (21,4%) — há registro de decisão, mas sem conteúdo decisório (ex: "Pleno em DD/MM/AAAA HH:MM"). Registro administrativo.
"decisão não capturada" (0,2%) — a falha residual identificada na Tarefa 2.

Não toca no desfecho original — a coluna nova é paralela, então nada quebra. Os gráficos existentes continuam funcionando; os novos podem usar a coluna refinada.

In [111]:
# ===========================================================================
# TAREFA 3 — Refinar o balde "motivos diversos" do Plenário Físico
# ===========================================================================
# Separa os "motivos diversos" em subcategorias transparentes:
#   - sem decisão registrada na janela (pauta que não virou julgamento)
#   - decisão registrada sem fórmula de votação (registro administrativo)
#   - decisão classificável não capturada (falha residual de detecção)

RE_EXCLUSAO = (
    r'Incluído na Lista|Agendado para|Julgamento Presencial|'
    r'Sess[ãa]o (?:Ordinária|Extraordinária|Ordinaria|Extraordinaria) Virtual'
)

dec_pleno = df_dec[
    (df_dec['dec_julgador'].str.strip().str.upper() == 'TRIBUNAL PLENO') &
    (~df_dec['dec_complemento'].str.contains(RE_EXCLUSAO, case=False, na=False, regex=True))
].copy()
dec_pleno['dec_data_dt'] = pd.to_datetime(
    dec_pleno['dec_data'], dayfirst=True, errors='coerce'
)


def subcategoria_motivos_diversos(row):
    """
    Refina o balde 'motivos diversos' do PP em três subcategorias.
    Retorna o desfecho original para os demais casos.
    """
    if row['ambiente'] != 'Plenário Físico':
        return row['desfecho']
    if row['desfecho'] != 'Não concluído - motivos diversos':
        return row['desfecho']

    inc = row['incidente']
    inicio = row['data_inclusao_dt']

    # Janela: até a próxima inclusão do mesmo incidente
    proximas = df_final[
        (df_final['incidente'] == inc) &
        (df_final['data_inclusao_dt'] > inicio)
    ]['data_inclusao_dt']
    prox_min = proximas.min() if not proximas.empty else pd.Timestamp('2026-12-31')
    # Cap em +7d: mesma janela padrão (fallback) usada pelo classificador
    # original (classificar_desfecho_fisico/extrair_janela_sessao). Sem esse
    # cap, a janela ia até a próxima inclusão em pauta (meses depois) e
    # pegava decisões de PLENO de um pautamento seguinte não relacionado,
    # inflando "decisão não capturada" de ~21 casos reais para 382.
    fim = min(inicio + pd.Timedelta(days=7), prox_min)

    dec_janela = dec_pleno[
        (dec_pleno['incidente'] == inc) &
        (dec_pleno['dec_data_dt'] >= inicio) &
        (dec_pleno['dec_data_dt'] < fim)
    ]

    if dec_janela.empty:
        return 'Não concluído - sem decisão registrada'

    classificaveis = [
        t for t in dec_janela['dec_complemento']
        if classificar_desfecho_texto(t) is not None
    ]
    if classificaveis:
        return 'Não concluído - decisão não capturada'   # falha residual (0,7%)
    return 'Não concluído - registro sem fórmula de votação'


print("Refinando o balde 'motivos diversos' do Plenário Físico...")
df_final['desfecho_refinado'] = df_final.apply(subcategoria_motivos_diversos, axis=1)

# Resultado
print(f"\n{'='*62}")
print(f"DESFECHO REFINADO — PLENÁRIO FÍSICO")
print(f"{'='*62}")
mask_pp = df_final['ambiente'] == 'Plenário Físico'
print(df_final.loc[mask_pp, 'desfecho_refinado'].value_counts().to_string())

print(f"\n{'='*62}")
print(f"COMPARAÇÃO — antes e depois do refinamento (PP)")
print(f"{'='*62}")
antes = (df_final.loc[mask_pp, 'desfecho'] == 'Não concluído - motivos diversos').sum()
print(f"  Antes: 'motivos diversos' = {antes:,} (balde único)")
print(f"  Depois:")
for cat in ['Não concluído - sem decisão registrada',
            'Não concluído - registro sem fórmula de votação',
            'Não concluído - decisão não capturada']:
    n = (df_final.loc[mask_pp, 'desfecho_refinado'] == cat).sum()
    print(f"    {cat:<50} {n:>5,} ({100*n/antes:>5.1f}%)")

Refinando o balde 'motivos diversos' do Plenário Físico...

DESFECHO REFINADO — PLENÁRIO FÍSICO
desfecho_refinado
Não concluído - sem decisão registrada             2607
Não concluído - registro sem fórmula de votação     724
Não concluído - retirado de pauta                   652
Não concluído - decisão não capturada               382
Concluído - decisão maioria com o relator           209
Concluído - decisão unânime                         160
Não concluído - pedido de vista                     104
Concluído - decisão maioria, vencido o relator       25

COMPARAÇÃO — antes e depois do refinamento (PP)
  Antes: 'motivos diversos' = 3,713 (balde único)
  Depois:
    Não concluído - sem decisão registrada             2,607 ( 70.2%)
    Não concluído - registro sem fórmula de votação      724 ( 19.5%)
    Não concluído - decisão não capturada                382 ( 10.3%)


### Tarefa 3.2 — desdobrando "sem decisão registrada" por ano (pedido da Jana)

Refinamento acima (`desfecho_refinado`) já separa "registro sem fórmula de
votação" (categoria 2 do pedido) e "decisão não capturada" (falha residual).
Falta separar o que sobra em "sem decisão registrada" em:

1. sem qualquer andamento posterior na janela (pautada e não julgada, sem mais nada)
2. *(já coberto acima)* registro sem fórmula de votação
3. andamento de retirada/adiamento/reagendamento (`ADIADO O JULGAMENTO`,
   `Suspenso o julgamento`, `Sobrestado`) — presentes na janela mas não
   capturados como top-level "retirado de pauta" nem como decisão
4. outros — resíduo (aqui, a própria "decisão não capturada")

Tabela por ano, PP, 2020–2025.

In [ ]:
# ===========================================================================
# TAREFA 3.2 — Categoria 1 vs 3 dentro de "sem decisão registrada" (PP)
# ===========================================================================
# Usa a mesma janela (cap de 7 dias / próxima inclusão) e o mesmo dec_pleno
# já calculados na Tarefa 3 acima. Só adiciona uma checagem: havia um
# andamento de retirada/adiamento/reagendamento (ANDAMENTOS_ADIAMENTO) na
# janela, mesmo sem decisão de mérito registrada?

from inclusao_pauta import ANDAMENTOS_ADIAMENTO


def subcategoria_sem_decisao(row):
    if row['desfecho_refinado'] != 'Não concluído - sem decisão registrada':
        return None

    inc = row['incidente']
    inicio = row['data_inclusao_dt']

    proximas = df_final[
        (df_final['incidente'] == inc) &
        (df_final['data_inclusao_dt'] > inicio)
    ]['data_inclusao_dt']
    prox_min = proximas.min() if not proximas.empty else pd.Timestamp('2026-12-31')
    fim = min(inicio + pd.Timedelta(days=7), prox_min)

    eventos = df_and[
        (df_and['incidente'] == inc) &
        (df_and['and_data_dt'] >= inicio) &
        (df_and['and_data_dt'] < fim)
    ]
    if eventos['and_nome'].isin(ANDAMENTOS_ADIAMENTO).any():
        return '3-Retirada/adiamento/reagendamento'
    return '1-Sem andamento posterior indicativo de julgamento'


mask_pp = df_final['ambiente'] == 'Plenário Físico'
df_final['subcategoria_pedido_jana'] = None
df_final.loc[mask_pp, 'subcategoria_pedido_jana'] = (
    df_final.loc[mask_pp].apply(subcategoria_sem_decisao, axis=1)
)
df_final.loc[
    mask_pp & (df_final['desfecho_refinado'] == 'Não concluído - registro sem fórmula de votação'),
    'subcategoria_pedido_jana'
] = '2-Decisão registrada sem fórmula de votação'
df_final.loc[
    mask_pp & (df_final['desfecho_refinado'] == 'Não concluído - decisão não capturada'),
    'subcategoria_pedido_jana'
] = '4-Decisão não capturada (falha residual)'

sub = df_final[
    mask_pp
    & df_final['subcategoria_pedido_jana'].notna()
    & df_final['ano'].between(2020, 2025)
].copy()

tabela_jana = sub.groupby(['ano', 'subcategoria_pedido_jana']).size().unstack(fill_value=0)
tabela_jana = tabela_jana[sorted(tabela_jana.columns)]
tabela_jana['Total'] = tabela_jana.sum(axis=1)
tabela_jana.loc['Total'] = tabela_jana.sum(axis=0)

print("Categorização de 'motivos diversos' — Plenário Físico, 2020–2025")
print(tabela_jana.to_string())

### Comparação com os resultados da dissertação

In [112]:
# Filtro de período
mask_periodo = (
    (df_final['data_inclusao_dt'] >= '2021-08-06') &
    (df_final['data_inclusao_dt'] <= '2021-12-17')
)
df_2021s2 = df_final[mask_periodo].copy()

In [ ]:
# ===========================================================================
# COMPARAÇÃO COM A TESE — 2º SEMESTRE JUDICIAL DE 2021 (06/08 a 17/12/2021)
# ===========================================================================

# ── Filtros de período para os dois datasets ─────────────────────────────
mask_periodo_pauta = (
    (df_final['data_inclusao_dt'] >= '2021-08-06') &
    (df_final['data_inclusao_dt'] <= '2021-12-17')
)
df_2021s2 = df_final[mask_periodo_pauta].copy()

mask_periodo_sessao = (
    (df_sessoes_final['data_sessao_dt'] >= '2021-08-06') &
    (df_sessoes_final['data_sessao_dt'] <= '2021-12-17')
)
df_sessoes_2021s2 = df_sessoes_final[mask_periodo_sessao].copy()

# ---------------------------------------------------------------------------
# 1 — Resumo geral: INCLUSÕES EM PAUTA
# ---------------------------------------------------------------------------
print(f"{'='*55}")
print(f"INCLUSÕES EM PAUTA — 2º SEMESTRE JUDICIAL DE 2021")
print(f"(06/08/2021 a 17/12/2021)")
print(f"{'='*55}")
print(f"Total de inclusões em pauta: {len(df_2021s2):,}")
print(f"Processos distintos:         {df_2021s2['incidente'].nunique():,}")

print(f"\nPor ambiente:")
print(df_2021s2['ambiente'].value_counts().to_string())

print(f"\nPor classe:")
print(df_2021s2['classe'].value_counts().to_string())

print(f"\nPor mês e ambiente:")
tab_mes_amb = (
    df_2021s2
    .assign(mes=df_2021s2['data_inclusao_dt'].dt.to_period('M'))
    .groupby(['mes', 'ambiente'])
    .size()
    .unstack(fill_value=0)
)
tab_mes_amb['TOTAL'] = tab_mes_amb.sum(axis=1)
print(tab_mes_amb.to_string())

# ---------------------------------------------------------------------------
# 2 — Resumo geral: SESSÕES VIRTUAIS INICIADAS
# ---------------------------------------------------------------------------
print(f"\n{'='*55}")
print(f"SESSÕES VIRTUAIS INICIADAS — 2º SEMESTRE JUDICIAL DE 2021")
print(f"(06/08/2021 a 17/12/2021)")
print(f"{'='*55}")
print(f"Total de sessões virtuais: {len(df_sessoes_2021s2):,}")
print(f"Processos distintos:       {df_sessoes_2021s2['incidente'].nunique():,}")

print(f"\nPor classe:")
print(df_sessoes_2021s2['classe'].value_counts().to_string())

print(f"\nPor mês:")
tab_mes_sessao = (
    df_sessoes_2021s2
    .assign(mes=df_sessoes_2021s2['data_sessao_dt'].dt.to_period('M'))
    .groupby('mes')
    .size()
)
print(tab_mes_sessao.to_string())

# ---------------------------------------------------------------------------
# 3 — Comparação lado a lado (só Virtual, que é o escopo da tese)
# ---------------------------------------------------------------------------
print(f"\n{'='*55}")
print(f"COMPARAÇÃO — PLENÁRIO VIRTUAL (2021.2)")
print(f"{'='*55}")

df_2021s2_virt = df_2021s2[df_2021s2['ambiente'] == 'Plenário Virtual']

print(f"{'Métrica':<35} {'Inclusões':>12} {'Sessões':>12}")
print(f"{'-'*60}")
print(f"{'Total de eventos':<35} {len(df_2021s2_virt):>12,} {len(df_sessoes_2021s2):>12,}")
print(f"{'Processos distintos':<35} {df_2021s2_virt['incidente'].nunique():>12,} {df_sessoes_2021s2['incidente'].nunique():>12,}")

# Tipo de questão
print(f"\nTipo de questão (Plenário Virtual):")
print(f"{'Tipo':<20} {'Inclusões':>12} {'Sessões':>12}")
print(f"{'-'*46}")
tipos = ['PR', 'RC', 'IJ']
for t in tipos:
    n_incl = (df_2021s2_virt['tipo_questao'] == t).sum()
    n_sess = (df_sessoes_2021s2['tipo_questao'] == t).sum()
    print(f"{t:<20} {n_incl:>12,} {n_sess:>12,}")
print("  Referência tese: ~2/3 PR, ~1/4 RC, <10% IJ")

# Desfecho
print(f"\nDesfecho (Plenário Virtual):")
print(f"{'Desfecho':<48} {'Incl.':>7} {'Sess.':>7}")
print(f"{'-'*64}")
todos_desfechos = sorted(
    set(df_2021s2_virt['desfecho'].unique()) |
    set(df_sessoes_2021s2['desfecho'].unique())
)
for d in todos_desfechos:
    n_incl = (df_2021s2_virt['desfecho'] == d).sum()
    n_sess = (df_sessoes_2021s2['desfecho'] == d).sum()
    print(f"{d:<48} {n_incl:>7,} {n_sess:>7,}")

INCLUSÕES EM PAUTA — 2º SEMESTRE JUDICIAL DE 2021
(06/08/2021 a 17/12/2021)
Total de inclusões em pauta: 633
Processos distintos:         442

Por ambiente:
ambiente
Plenário Virtual    443
Plenário Físico     190

Por classe:
classe
ADI     491
ADPF    126
ADC      16
ADO       0

Por mês e ambiente:
ambiente  Plenário Físico  Plenário Virtual  TOTAL
mes                                               
2021-08                10                83     93
2021-09                 9               121    130
2021-10                36               111    147
2021-11                28                93    121
2021-12               107                35    142

SESSÕES VIRTUAIS INICIADAS — 2º SEMESTRE JUDICIAL DE 2021
(06/08/2021 a 17/12/2021)
Total de sessões virtuais: 424
Processos distintos:       363

Por classe:
classe
ADI     335
ADPF     76
ADC      12
ADO       1

Por mês:
mes
2021-08     78
2021-09     89
2021-10    137
2021-11     66
2021-12     54
Freq: M

COMPARAÇÃO — PLENÁRIO VIRTU

### Visualização específica

#### Amostra de inclusões virtuais aleatórias completas

In [114]:
# Amostra de 10 inclusões virtuais com detalhes completos
amostra = (
    df_final[df_final['ambiente'] == 'Plenário Virtual']
    .sample(10, random_state=42)
    [[
        'nome_processo', 'classe', 'relator',
        'data_inclusao', 'tipo_questao', 'sufixo_extraido', 'desfecho'
    ]]
    .sort_values('data_inclusao')
    .reset_index(drop=True)
)

# Configura o pandas para mostrar tudo sem cortar
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print(amostra.to_string())

  nome_processo classe               relator data_inclusao tipo_questao                sufixo_extraido                                   desfecho
0      ADI 7651    ADI              LUIZ FUX    04/02/2025           IJ                     ADI-MC-Ref            Não concluído - pedido de vista
1      ADPF 982   ADPF           FLÁVIO DINO    05/02/2025           PR                           None                Concluído - decisão unânime
2      ADI 6195    ADI   ALEXANDRE DE MORAES    14/02/2020           PR                           None          Não concluído - retirado de pauta
3      ADI 7708    ADI           FLÁVIO DINO    20/02/2025           IJ                     ADI-MC-Ref            Não concluído - pedido de vista
4      ADI 6384    ADI  LUÍS ROBERTO BARROSO    20/06/2023           PR                           None                   Não concluído - destaque
5      ADI 6872    ADI         GILMAR MENDES    21/09/2021           PR                            ADI            Não conclu

In [115]:

# Versão corrigida: respeita a janela de cada entrada para mostrar a decisão correta
amostra = df_final[df_final['ambiente'] == 'Plenário Virtual'].sample(10, random_state=42)

for _, pauta in amostra.iterrows():
    inc = pauta['incidente']
    dt_pauta = pauta['data_inclusao_dt']
    comp_pauta = pauta.get('andamento_origem', '') or ''

    # 1. Calcula a mesma janela usada no algoritmo de classificação
    sessao_inicio, sessao_fim = extrair_janela_sessao(comp_pauta, dt_pauta)

    # Busca a data da próxima entrada para fechar a janela
    proxima = df_final[
        (df_final['incidente'] == inc) &
        (df_final['data_inclusao_dt'] > dt_pauta)
    ]['data_inclusao_dt'].min()

    fim_janela = min(proxima, sessao_fim) if pd.notna(proxima) and proxima > sessao_inicio else sessao_fim

    print("=" * 80)
    print(f"Processo:   {pauta['nome_processo']} ({pauta['classe']})")
    print(f"Inclusão:   {pauta['data_inclusao']}")
    print(f"Janela:     {sessao_inicio.strftime('%d/%m/%Y')} a {fim_janela.strftime('%d/%m/%Y')}")
    print(f"Desfecho:   {pauta['desfecho']}")

    # 2. Busca decisões APENAS dentro desta janela específica
    dec = dec_virt[
        (dec_virt['incidente'] == inc) &
        (dec_virt['dec_data_dt'] >= sessao_inicio - pd.Timedelta(days=1)) &
        (dec_virt['dec_data_dt'] < fim_janela)
    ].copy()

    if not dec.empty:
        dec['diff'] = (dec['dec_data_dt'] - sessao_inicio).dt.days.abs()
        dec = dec.sort_values('diff')

        # Tenta pegar a primeira que seja classificável
        texto_final = "(Texto não classificável encontrado na janela)"
        for _, d in dec.iterrows():
            texto = d['dec_complemento']
            if classificar_desfecho_texto(texto):
                texto_final = texto
                break
        print(f"Decisão:    {texto_final[:350]}...")
    else:
        # Se não há decisão na janela, o desfecho veio de um andamento (Vista/Destaque/Retirada)
        # Vamos buscar o andamento que justificou isso
        eventos_janela = df_and_virt[
            (df_and_virt['incidente'] == inc) &
            (df_and_virt['and_data_dt'] >= dt_pauta) &
            (df_and_virt['and_data_dt'] < fim_janela) &
            (df_and_virt['and_index'] > pauta.get('and_index', 0))
        ]

        if not eventos_janela.empty:
            # Pega o primeiro andamento relevante que casou com a regra
            for _, ev in eventos_janela.iterrows():
                if ev['and_nome'] in ANDAMENTOS_RETIRADA or ev['and_nome'] in DESTAQUE_NOMES or eh_vista_ministro(ev['and_nome'], ev['and_complemento']):
                    print(f"Evento:     [{ev['and_data']}] {ev['and_nome']} - {ev['and_complemento']}")
                    break
        else:
            print(f"Decisão:    (Nenhuma decisão ou evento de saída encontrado na janela)")
    print()

Processo:   ADI 7708 (ADI)
Inclusão:   20/02/2025
Janela:     20/02/2025 a 22/03/2025
Desfecho:   Não concluído - pedido de vista
Evento:     [17/03/2025] Vista ao(à) Ministro(a) - Decisão: Após o voto-vista do Ministro Luís Roberto Barroso (Presidente), que não referendava a medida cautelar, no que foi acompanhado pelos Ministros Gilmar Mendes, Cristiano Zanin e André Mendonça; e do voto do Ministro Dias Toffoli, que acompanhava o Ministro Flávio Dino (Relator), pediu vista dos autos o Ministro Alexandre de Moraes. Plenário, Sessão Virtual de 7.3.2025 a 14.3.2025.

Processo:   ADI 7651 (ADI)
Inclusão:   04/02/2025
Janela:     04/02/2025 a 06/03/2025
Desfecho:   Não concluído - pedido de vista
Evento:     [24/02/2025] Vista ao(à) Ministro(a) - Decisão: Após o voto do Ministro Luiz Fux (Relator), que propunha o referendo da decisão que deferiu parcialmente a medida cautelar, para se conferir ao inciso III do parágrafo nono do artigo 136 e ao artigo 136-A, da Constituição do Estado do Ma

#### Visualizar processo específico

In [116]:
# --- VARIÁVEIS DE BUSCA ---
CLASSE_BUSCA = 'ADI'
NUMERO_BUSCA = 1057
# --------------------------

In [117]:
nome_alvo = f"{CLASSE_BUSCA} {NUMERO_BUSCA}"
resultado_busca = df_final[df_final['nome_processo'] == nome_alvo].copy()

if resultado_busca.empty:
    print(f"Nenhum registro encontrado para: {nome_alvo}")
else:
    inc = resultado_busca['incidente'].iloc[0]

    # -----------------------------------------------------------------------
    # 1 — Resumo do processo
    # -----------------------------------------------------------------------
    print(f"{'='*65}")
    print(f"PROCESSO: {nome_alvo}")
    print(f"{'='*65}")
    print(f"Incidente:  {inc}")
    print(f"Classe:     {resultado_busca['classe'].iloc[0]}")
    print(f"Relator:    {resultado_busca['relator'].iloc[0]}")
    print(f"Total de inclusões em pauta: {len(resultado_busca)}")
    print(f"  Plenário Virtual: {(resultado_busca['ambiente'] == 'Plenário Virtual').sum()}")
    print(f"  Plenário Físico:  {(resultado_busca['ambiente'] == 'Plenário Físico').sum()}")

    # Desfecho final do processo
    g = resultado_busca.sort_values('data_inclusao_dt')
    concluidas = g[g['macro_desfecho'] == 'Concluído']
    desf_final = concluidas.iloc[-1]['desfecho'] if not concluidas.empty else g.iloc[-1]['desfecho']
    print(f"\nDesfecho final do processo: {desf_final}")

    # -----------------------------------------------------------------------
    # 2 — Tabela de todas as entradas em pauta
    # -----------------------------------------------------------------------
    print(f"\n{'='*65}")
    print(f"TODAS AS ENTRADAS EM PAUTA (cronológica)")
    print(f"{'='*65}")
    display(
        resultado_busca.sort_values('data_inclusao_dt')[[
            'data_inclusao', 'ambiente', 'tipo_questao',
            'sufixo_extraido', 'desfecho', 'macro_desfecho'
        ]].reset_index(drop=True)
    )

    # -----------------------------------------------------------------------
    # 3 — Andamentos relevantes do processo (entradas, saídas, desfechos)
    # -----------------------------------------------------------------------
    ANDAMENTOS_RELEVANTES = (
        set(ANDAMENTOS_PAUTA) |
        ANDAMENTOS_RETIRADA |
        DESTAQUE_NOMES |
        SESSION_VISTA_NAMES |
        {
            'Iniciado Julgamento Virtual',
            'Finalizado Julgamento Virtual',
            'Suspenso o julgamento',
            'ADIADO O JULGAMENTO',
            'Adiado o julgamento',
        }
    )

    and_proc = df_and[df_and['incidente'] == inc].copy()
    and_proc['and_data_dt'] = pd.to_datetime(
        and_proc['and_data'], dayfirst=True, errors='coerce'
    )
    and_relevantes = and_proc[
        and_proc['and_nome'].isin(ANDAMENTOS_RELEVANTES)
    ].sort_values('and_data_dt')

    print(f"\n{'='*65}")
    print(f"ANDAMENTOS RELEVANTES (entradas, saídas e eventos de julgamento)")
    print(f"{'='*65}")
    if and_relevantes.empty:
        print("  (nenhum andamento relevante encontrado)")
    else:
        for _, row in and_relevantes.iterrows():
            data = row['and_data_dt'].strftime('%d/%m/%Y') if pd.notna(row['and_data_dt']) else '?'
            comp = str(row['and_complemento'] or '')
            comp_resumo = comp[:100] + '...' if len(comp) > 100 else comp
            print(f"\n  [{data}] {row['and_nome']}")
            if comp_resumo.strip():
                print(f"           {comp_resumo}")

    # -----------------------------------------------------------------------
    # ── Detalhamento — Plenário Virtual (com janela real da sessão) ───────────
print("\n--- Detalhamento das Decisões (Plenário Virtual) ---")

pautas_pv = resultado_busca[
    resultado_busca['ambiente'] == 'Plenário Virtual'
].sort_values('data_inclusao_dt').copy()

for _, pauta in pautas_pv.iterrows():
    dt_pauta = pauta['data_inclusao_dt']
    comp     = pauta.get('andamento_origem', '') or ''

    # Usa a mesma lógica da classificação
    sessao_inicio, sessao_fim = extrair_janela_sessao(comp, dt_pauta)

    # Respeita a próxima entrada como limite superior
    pautas_futuras = pautas_pv[pautas_pv['data_inclusao_dt'] > dt_pauta]
    proxima = pautas_futuras['data_inclusao_dt'].min() if not pautas_futuras.empty else pd.NaT
    fim_janela = min(proxima, sessao_fim) if pd.notna(proxima) and proxima < sessao_fim else sessao_fim

    print(f"\n  Pauta de {pauta['data_inclusao']}  |  Desfecho: {pauta['desfecho']}")
    print(f"  Tipo: {pauta['tipo_questao']}  |  Sufixo: {pauta['sufixo_extraido']}")
    print(f"  Janela de busca: {sessao_inicio.strftime('%d/%m/%Y')} → {fim_janela.strftime('%d/%m/%Y')}")

    dec_match = dec_virt[
        (dec_virt['incidente'] == inc) &
        (dec_virt['dec_data_dt'] >= sessao_inicio - pd.Timedelta(days=1)) &
        (dec_virt['dec_data_dt'] < fim_janela)
    ].copy()

    if dec_match.empty:
        print("  Decisão: (nenhuma decisão virtual dentro da janela)")
    else:
        dec_match['diff'] = (dec_match['dec_data_dt'] - sessao_inicio).dt.days.abs()
        dec_match = dec_match.sort_values('diff')

        texto_dec = None
        for _, d in dec_match.iterrows():
            if classificar_desfecho_texto(d['dec_complemento']) is not None:
                texto_dec = d['dec_complemento']
                break

        if texto_dec is None:
            print("  Decisão: (desfecho veio de andamento — retirada/vista/destaque)")
        else:
            print(f"  Decisão: {texto_dec[:400] if isinstance(texto_dec, str) else '(sem texto)'}")

    print(f"  {'-'*60}")

    # -----------------------------------------------------------------------
    # 5 — Detalhamento por entrada física (decisão mais próxima)
    # -----------------------------------------------------------------------
    pautas_pp = resultado_busca[
        resultado_busca['ambiente'] == 'Plenário Físico'
    ].sort_values('data_inclusao_dt').copy()

    if len(pautas_pp) > 0:
        print(f"\n{'='*65}")
        print(f"DETALHAMENTO — PLENÁRIO FÍSICO")
        print(f"{'='*65}")

        dec_fisica = df_dec[
            (df_dec['incidente'] == inc) &
            (~df_dec['dec_julgador'].str.contains('VIRTUAL', case=False, na=False))
        ].copy()
        dec_fisica['dec_data_dt'] = pd.to_datetime(
            dec_fisica['dec_data'], dayfirst=True, errors='coerce'
        )
        dec_fisica = dec_fisica[
            ~dec_fisica['dec_complemento'].str.contains(
                r'Incluído na Lista|Agendado para|Julgamento Virtual',
                case=False, na=False, regex=True
            )
        ]

        pautas_pp['proxima_entrada_dt'] = pautas_pp['data_inclusao_dt'].shift(-1)

        for _, pauta in pautas_pp.iterrows():
            dt_pauta = pauta['data_inclusao_dt']
            proxima  = pauta['proxima_entrada_dt']
            fim_janela = proxima if pd.notna(proxima) else dt_pauta + pd.Timedelta(days=30)

            print(f"\n  Pauta de {pauta['data_inclusao']}")
            print(f"  Desfecho:   {pauta['desfecho']}")
            print(f"  Janela de busca: {dt_pauta.strftime('%d/%m/%Y')} → {fim_janela.strftime('%d/%m/%Y')}")

            cand = dec_fisica[
                (dec_fisica['dec_data_dt'] >= dt_pauta - pd.Timedelta(days=1)) &
                (dec_fisica['dec_data_dt'] < fim_janela)
            ].copy()

            if cand.empty:
                print(f"  Decisão: (nenhuma decisão física dentro da janela)")
            else:
                cand['diff'] = (cand['dec_data_dt'] - dt_pauta).dt.days.abs()
                melhor = cand.sort_values('diff').iloc[0]
                data_dec = melhor['dec_data_dt'].strftime('%d/%m/%Y') if pd.notna(melhor['dec_data_dt']) else '?'
                texto = str(melhor['dec_complemento'] or '')
                print(f"  Decisão ({data_dec}): {texto[:300]}")

            print(f"  {'-'*60}")

PROCESSO: ADI 1057
Incidente:  1585592
Classe:     ADI
Relator:    DIAS TOFFOLI
Total de inclusões em pauta: 2
  Plenário Virtual: 1
  Plenário Físico:  1

Desfecho final do processo: Concluído - decisão unânime

TODAS AS ENTRADAS EM PAUTA (cronológica)


,data_inclusao,ambiente,tipo_questao,sufixo_extraido,desfecho,macro_desfecho
0,12/09/2018,Plenário Físico,Não identificado,None,Não concluído - motivos diversos,Não concluído
1,28/06/2021,Plenário Virtual,PR,ADI,Concluído - decisão unânime,Concluído



ANDAMENTOS RELEVANTES (entradas, saídas e eventos de julgamento)

  [14/04/1994] VISTA AO MINISTRO
           MARCO AURÉLIO, DECISÃO: ADIADO O JULGAMENTO, EM FACE DO PEDIDO DE VISTA DO MIN. MARCO AURÉLIO, DEPOI...

  [12/09/2018] Inclua-se em pauta - minuta extraída
           Pleno em 12/09/2018 21:07:44 -

  [28/06/2021] Inclua-se em pauta - minuta extraída
           Julgamento Virtual: ADI. Incluído na Lista 264-2021.DT - Agendado para: 06/08/2021 a 16/08/2021.

  [28/06/2021] Retirado de pauta

  [06/08/2021] Iniciado Julgamento Virtual

  [17/08/2021] Finalizado Julgamento Virtual
           Finalizado Julgamento Virtual em 16 de Agosto de 2021 (Segunda-feira), às 23:59 .

--- Detalhamento das Decisões (Plenário Virtual) ---

  Pauta de 28/06/2021  |  Desfecho: Concluído - decisão unânime
  Tipo: PR  |  Sufixo: ADI
  Janela de busca: 28/06/2021 → 28/07/2021
  Decisão: (nenhuma decisão virtual dentro da janela)
  ------------------------------------------------------------

DETAL

## 6. Visualizações


Com a tabela final construída e validada, realizamos as análises
solicitadas, segmentando e correlacionando as variáveis.

In [118]:
CORES_CLASSE = {
    'ADI':  '#2563eb',
    'ADPF': '#f59e0b',
    'ADC':  '#16a34a',
    'ADO':  '#ef4444',
}

COR_TOTAL  = '#3498db'
COR_LINHA  = '#7f7f7f'

CORES_MACRO = {
    'Concluído':     '#16a34a',
    'Não concluído': '#ef4444',
    'Sem registro':  '#94a3b8',
}

LAYOUT_BASE = dict(
    template='plotly_white',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    legend=dict(
        orientation='h', yanchor='top', y=-0.18,
        xanchor='center', x=0.5,
        font=dict(size=10), bgcolor='#fcfcfc',
        bordercolor='#cccccc', borderwidth=1
    ),
)

In [119]:
# Mapeamento de desfecho para macro (concluído / não concluído)
def macro_desfecho(d):
    if str(d).startswith('Concluído'):
        return 'Concluído'
    if str(d).startswith('Não concluído'):
        return 'Não concluído'
    return 'Sem registro'

df_final['macro_desfecho'] = df_final['desfecho'].apply(macro_desfecho)

In [120]:
def plotar_barras_stf(df_dados, col_x, col_y, col_grupo=None,
                      titulo='', label_y='Inclusões em pauta',
                      mostrar_linha_total=False, df_total=None,
                      cores=None):
    """
    Função base de plotagem padronizada.

    Parâmetros:
      df_dados         : DataFrame já filtrado (ambiente, classe, etc.)
      col_x            : coluna do eixo X (ex: 'ano')
      col_y            : coluna de valores (ex: 'n')
      col_grupo        : coluna de agrupamento para barras empilhadas (ex: 'classe')
      titulo           : título do gráfico
      label_y          : rótulo do eixo Y
      mostrar_linha_total : se True, adiciona linha cinza com total geral
      df_total         : DataFrame com totais para a linha (colunas: col_x, col_y)
      cores            : dict {grupo: cor} ou None (usa padrão)
    """
    fig = make_subplots(specs=[[{'secondary_y': mostrar_linha_total}]])

    if col_grupo:
        grupos = df_dados[col_grupo].unique()
        for grupo in grupos:
            d = df_dados[df_dados[col_grupo] == grupo]
            cor = (cores or CORES_CLASSE).get(grupo, COR_TOTAL)
            fig.add_trace(go.Bar(
                x=d[col_x], y=d[col_y],
                name=grupo,
                marker_color=cor,
                text=d[col_y],
                textposition='outside',
                cliponaxis=False,
            ), secondary_y=False)
        fig.update_layout(barmode='group')
    else:
        cor = cores if isinstance(cores, str) else COR_TOTAL
        fig.add_trace(go.Bar(
            x=df_dados[col_x], y=df_dados[col_y],
            name=label_y,
            marker_color=cor,
            text=df_dados[col_y],
            textposition='outside',
            cliponaxis=False,
        ), secondary_y=False)

    if mostrar_linha_total and df_total is not None:
        fig.add_trace(go.Scatter(
            x=df_total[col_x], y=df_total[col_y],
            mode='lines+markers',
            line=dict(color=COR_LINHA, width=2),
            marker=dict(size=5),
            name='Total Geral',
        ), secondary_y=True)
        fig.update_yaxes(title_text='Total Geral (Linha)', secondary_y=True)

    fig.update_layout(
        title_text=titulo,
        template='plotly_white',
        margin=dict(t=80, b=120),
        legend=dict(
            orientation='h', yanchor='top', y=-0.18,
            xanchor='center', x=0.5,
            font=dict(size=10, color='#333333'),
            bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1
        ),
        xaxis=dict(dtick=1, title='Ano'),
        yaxis=dict(title=label_y),
    )
    fig.show()
    return fig

In [121]:
def plotar_pizza_stf(series, titulo='', buraco=0.4, cores=None):
    """
    Gráfico de pizza (ou rosca) padronizado com suporte a cores.
    """
    fig = go.Figure(go.Pie(
        labels=series.index,
        values=series.values,
        hole=buraco,
        textinfo='label+percent+value',
        textposition='outside',
        marker=dict(colors=[cores.get(label) for label in series.index] if cores else None,
                    line=dict(color='white', width=2)),
    ))
    fig.update_layout(
        title_text=titulo,
        template='plotly_white',
        margin=dict(t=80, b=80),
        legend=dict(
            orientation='h', yanchor='top', y=-0.1,
            xanchor='center', x=0.5,
            font=dict(size=10),
        ),
        showlegend=True,
    )
    fig.show()
    return fig

### Inclusões em pauta. Geral. Anual. PV e PP.

**O que mostra**: volume total de inclusões em pauta por ano (2020–2025),
comparando Plenário Virtual e Plenário Físico lado a lado.

**Como foi calculado**: agrupamento de `df_final` por `ano` e `ambiente`,
contando o número de linhas (cada linha = 1 inclusão em pauta).


In [122]:
tab5 = (
    df_final.groupby(['ano', 'ambiente'])
    .size().reset_index(name='n')
)

fig5 = make_subplots(specs=[[{'secondary_y': False}]])
for amb, cor in [('Plenário Virtual', '#2563eb'), ('Plenário Físico', '#94a3b8')]:
    d = tab5[tab5['ambiente'] == amb]
    fig5.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=amb,
        marker_color=cor,
        text=d['n'], textposition='outside', cliponaxis=False,
    ))
fig5.update_layout(
    title_text='Gráfico 5 — Inclusões em Pauta por Ano e Ambiente (2020–2025)',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Inclusões em pauta'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)
fig5.show()

# Pizza complementar: proporção PV vs PP no período total
# Atribuímos a uma variável para evitar a exibição duplicada no Colab
pizza5 = df_final['ambiente'].value_counts()
fig5b = plotar_pizza_stf(pizza5, titulo='Gráfico 5b — Proporção PV vs PP (período total)')

### Inclusões em pauta. Por classe. Anual. PV.

**O que mostra**: volume de inclusões em pauta no Plenário Virtual por
ano e por classe processual (ADI, ADPF, ADC, ADO).

**Como foi calculado**: filtramos `df_final` para `ambiente == Plenário Virtual`,
depois agrupamos por `ano` e `classe`.

In [123]:
valor_maximo = 1000

In [124]:
df_pv = df_final[df_final['ambiente'] == 'Plenário Virtual']
df_pp = df_final[df_final['ambiente'] == 'Plenário Físico']

tab6 = df_pv.groupby(['ano', 'classe']).size().reset_index(name='n')
total6 = df_pv.groupby('ano').size().reset_index(name='n')

fig6 = make_subplots(specs=[[{'secondary_y': True}]])
for classe in ['ADI', 'ADPF', 'ADC', 'ADO']:
    d = tab6[tab6['classe'] == classe]
    fig6.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=classe,
        marker_color=CORES_CLASSE.get(classe, COR_TOTAL),
        text=d['n'], textposition='outside', cliponaxis=False,
    ), secondary_y=False)
fig6.add_trace(go.Scatter(
    x=total6['ano'], y=total6['n'],
    mode='lines+markers', line=dict(color=COR_LINHA, width=2),
    marker=dict(size=5), name='Total PV',
), secondary_y=True)
fig6.update_layout(
    title_text='Gráfico 6 — Inclusões em Pauta por Classe e Ano — Plenário Virtual',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Inclusões (por classe)'),
    yaxis2=dict(title='Total PV (Linha)'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)

fig6.update_yaxes(range=[0, valor_maximo])   # eixo esquerdo
fig6.update_yaxes(range=[0, valor_maximo], secondary_y=True)   # eixo direito

fig6.show()

pizza6 = df_pv['classe'].value_counts()
# Atribuímos a uma variável para evitar a exibição duplicada no Colab
fig6b = plotar_pizza_stf(pizza6, titulo='Gráfico 6b — Proporção por Classe — PV (período total)')

/tmp/ipykernel_960/584468226.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



### Inclusões em pauta. Por classe. Anual. PP.

Idêntico ao Gráfico anterior, mas para o Plenário Físico.

In [125]:
tab7 = df_pp.groupby(['ano', 'classe']).size().reset_index(name='n')
total7 = df_pp.groupby('ano').size().reset_index(name='n')

fig7 = make_subplots(specs=[[{'secondary_y': True}]])
for classe in ['ADI', 'ADPF', 'ADC', 'ADO']:
    d = tab7[tab7['classe'] == classe]
    fig7.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=classe,
        marker_color=CORES_CLASSE.get(classe, COR_TOTAL),
        text=d['n'], textposition='outside', cliponaxis=False,
    ), secondary_y=False)
fig7.add_trace(go.Scatter(
    x=total7['ano'], y=total7['n'],
    mode='lines+markers', line=dict(color=COR_LINHA, width=2),
    marker=dict(size=5), name='Total PP',
), secondary_y=True)
fig7.update_layout(
    title_text='Gráfico 7 — Inclusões em Pauta por Classe e Ano — Plenário Físico',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Inclusões (por classe)'),
    yaxis2=dict(title='Total PP (Linha)'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)

fig7.update_yaxes(range=[0, valor_maximo])   # eixo esquerdo
fig7.update_yaxes(range=[0, valor_maximo], secondary_y=True)   # eixo direito

fig7.show()

pizza7 = df_pp['classe'].value_counts()
# Atribuímos a uma variável para evitar a exibição duplicada no Colab
fig7b = plotar_pizza_stf(pizza7, titulo='Gráfico 7b — Proporção por Classe — PP (período total)')

/tmp/ipykernel_960/669189526.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



### Concluídos e Não Concluídos. Período Total. PV e PP.

**O que mostra**: proporção de pautas concluídas vs não concluídas no
período total (2020–2025), separado por ambiente.

**Como foi calculado**: usamos a coluna `macro_desfecho` (derivada de
`desfecho`) que agrupa em "Concluído", "Não concluído" e "Sem registro".

In [126]:
pizza8 = df_pv['macro_desfecho'].value_counts()
# Atribuímos a uma variável para evitar a exibição duplicada no Colab
fig8 = plotar_pizza_stf(pizza8, titulo='Gráfico 8 — Concluídos e Não Concluídos — PV (período total)')

In [127]:
# Desfecho detalhado PV
pizza8b = df_pv['desfecho'].value_counts()
fig8b = plotar_pizza_stf(pizza8b, titulo='Gráfico 8b — Desfecho Detalhado — PV (período total)', buraco=0.3)

In [128]:
# Gráfico 9 — PP, período total
pizza9 = df_pp['macro_desfecho'].value_counts()
fig9 = plotar_pizza_stf(pizza9, titulo='Gráfico 9 — Concluídos e Não Concluídos — PP (período total)')

### Concluídos e Não Concluídos. Anual. PV e PP.

**O que mostra**: evolução anual do volume de pautas concluídas vs não
concluídas, separado por ambiente.

**Como foi calculado**: agrupamento por `ano` e `macro_desfecho`, contando
o número de inclusões em cada combinação.

In [129]:
CORES_MACRO = {'Concluído': '#16a34a', 'Não concluído': '#ef4444', 'Sem registro': '#94a3b8'}

# Gráfico 10 — PV anual
tab10 = df_pv.groupby(['ano', 'macro_desfecho']).size().reset_index(name='n')
fig10 = make_subplots(specs=[[{'secondary_y': False}]])
for macro in ['Concluído', 'Não concluído', 'Sem registro']:
    d = tab10[tab10['macro_desfecho'] == macro]
    if len(d) == 0:
        continue
    fig10.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=macro,
        marker_color=CORES_MACRO.get(macro, '#999'),
        text=d['n'], textposition='outside', cliponaxis=False,
    ))
fig10.update_layout(
    title_text='Gráfico 10 — Concluídos e Não Concluídos por Ano — PV',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Inclusões em pauta'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)
fig10.show()

In [130]:
# Gráfico 11 — PP anual
tab11 = df_pp.groupby(['ano', 'macro_desfecho']).size().reset_index(name='n')
fig11 = make_subplots(specs=[[{'secondary_y': False}]])
for macro in ['Concluído', 'Não concluído', 'Sem registro']:
    d = tab11[tab11['macro_desfecho'] == macro]
    if len(d) == 0:
        continue
    fig11.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=macro,
        marker_color=CORES_MACRO.get(macro, '#999'),
        text=d['n'], textposition='outside', cliponaxis=False,
    ))
fig11.update_layout(
    title_text='Gráfico 11 — Concluídos e Não Concluídos por Ano — PP',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Inclusões em pauta'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)
fig11.show()

### Apenas Concluídos. Anual. PV e PP.

**O que mostra**: volume anual de pautas **concluídas** (excluindo não
concluídas), separado por ambiente.

**Como foi calculado**: filtramos `macro_desfecho == 'Concluído'` e
agrupamos por ano.

In [131]:
# Gráfico 12 — PV, só concluídos
tab12 = (
    df_pv[df_pv['macro_desfecho'] == 'Concluído']
    .groupby('ano').size().reset_index(name='n')
)

# Atribuímos a uma variável para evitar a exibição duplicada no Colab
fig12 = plotar_barras_stf(
    tab12, col_x='ano', col_y='n',
    titulo='Gráfico 12 — Concluídos por Ano — PV',
    label_y='Inclusões concluídas',
    cores='#16a34a',
)

In [132]:
# Gráfico 13 — PP, só concluídos
tab13 = (
    df_pp[df_pp['macro_desfecho'] == 'Concluído']
    .groupby('ano').size().reset_index(name='n')
)
plotar_barras_stf(
    tab13, col_x='ano', col_y='n',
    titulo='Gráfico 13 — Concluídos por Ano — PP',
    label_y='Inclusões concluídas',
    cores='#16a34a',
)

In [133]:
print(df_pp['macro_desfecho'].value_counts())
print(df_pp['desfecho'].value_counts())

macro_desfecho
Não concluído    4469
Concluído         394
Name: count, dtype: int64
desfecho
Não concluído - motivos diversos                  3713
Não concluído - retirado de pauta                  652
Concluído - decisão maioria com o relator          209
Concluído - decisão unânime                        160
Não concluído - pedido de vista                    104
Concluído - decisão maioria, vencido o relator      25
Name: count, dtype: int64


In [134]:
# Pega os incidentes do PP
incidentes_pp = df_pp['incidente'].unique()

# Busca decisões desses incidentes que NÃO sejam do Plenário Virtual
dec_pp = df_dec[
    df_dec['incidente'].isin(incidentes_pp) &
    ~df_dec['dec_julgador'].str.contains('VIRTUAL|SESSAO VIRTUAL', case=False, na=False)
]

print(f"Decisões encontradas para incidentes do PP: {len(dec_pp):,}")
print(f"\nTop 10 valores de dec_julgador:")
print(dec_pp['dec_julgador'].value_counts().head(10).to_string())

print(f"\nExemplos de dec_complemento:")
for t in dec_pp['dec_complemento'].dropna().head(5):
    print(f"  {t[:200]!r}")

Decisões encontradas para incidentes do PP: 7,756

Top 10 valores de dec_julgador:
dec_julgador
TRIBUNAL PLENO               1996
MIN. ALEXANDRE DE MORAES      891
MIN. LUÍS ROBERTO BARROSO     708
MIN. GILMAR MENDES            688
MIN. DIAS TOFFOLI             602
MIN. LUIZ FUX                 428
MIN. MARCO AURÉLIO            382
MIN. EDSON FACHIN             367
MIN. FLÁVIO DINO              307
MIN. NUNES MARQUES            263

Exemplos de dec_complemento:
  'o pedido de Anildo Fabio de Araujo e DEFERIDO quanto aos demais. Publique-se.'
  'Decisão: Decisão: O Tribunal, por maioria e nos termos do voto do Relator, resolveu questão de ordem no sentido de prorrogar o prazo da liminar concedida, vencido o Senhor Ministro Marco Aurélio. Voto'
  'Decisão: O Tribunal, por unanimidade, deferiu a cautelar, nos termos do voto do Relator. Votou o Presidente. Falou pelo requerente a Dra. Maria Dolores Serra M. Martins. Ausente, justificadamente, nes'
  '19/09/2024 16:16:27 - Julgamento Virtua

### Não Concluídos por Classe. Anual. PV e PP.

**O que mostra**: volume anual de pautas **não concluídas** por classe
processual, separado por ambiente.

**Como foi calculado**: filtramos `macro_desfecho == 'Não concluído'`,
depois agrupamos por `ano` e `classe`.

In [135]:
tab14 = (
    df_pv[df_pv['macro_desfecho'] == 'Não concluído']
    .groupby(['ano', 'classe']).size().reset_index(name='n')
)
total14 = (
    df_pv[df_pv['macro_desfecho'] == 'Não concluído']
    .groupby('ano').size().reset_index(name='n')
)

fig14 = make_subplots(specs=[[{'secondary_y': True}]])
for classe in ['ADI', 'ADPF', 'ADC', 'ADO']:
    d = tab14[tab14['classe'] == classe]
    fig14.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=classe,
        marker_color=CORES_CLASSE.get(classe, COR_TOTAL),
        text=d['n'], textposition='outside', cliponaxis=False,
    ), secondary_y=False)
fig14.add_trace(go.Scatter(
    x=total14['ano'], y=total14['n'],
    mode='lines+markers', line=dict(color=COR_LINHA, width=2),
    marker=dict(size=5), name='Total Não Concluídos PV',
), secondary_y=True)
fig14.update_layout(
    title_text='Gráfico 14 — Não Concluídos por Classe e Ano — PV',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Não concluídos (por classe)'),
    yaxis2=dict(title='Total Não Concluídos PV'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)
fig14.show()

/tmp/ipykernel_960/1768132953.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [136]:
# Gráfico 15 — PP, não concluídos por classe
tab15 = (
    df_pp[df_pp['macro_desfecho'] == 'Não concluído']
    .groupby(['ano', 'classe']).size().reset_index(name='n')
)
total15 = (
    df_pp[df_pp['macro_desfecho'] == 'Não concluído']
    .groupby('ano').size().reset_index(name='n')
)

fig15 = make_subplots(specs=[[{'secondary_y': True}]])
for classe in ['ADI', 'ADPF', 'ADC', 'ADO']:
    d = tab15[tab15['classe'] == classe]
    if len(d) == 0:
        continue
    fig15.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=classe,
        marker_color=CORES_CLASSE.get(classe, COR_TOTAL),
        text=d['n'], textposition='outside', cliponaxis=False,
    ), secondary_y=False)
fig15.add_trace(go.Scatter(
    x=total15['ano'], y=total15['n'],
    mode='lines+markers', line=dict(color=COR_LINHA, width=2),
    marker=dict(size=5), name='Total Não Concluídos PP',
), secondary_y=True)
fig15.update_layout(
    title_text='Gráfico 15 — Não Concluídos por Classe e Ano — PP',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Não concluídos (por classe)'),
    yaxis2=dict(title='Total Não Concluídos PP'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)
fig15.show()

/tmp/ipykernel_960/1790410926.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



### Concluídos por Classe. Anual. PV e PP.

**O que mostra**: volume anual de pautas **concluídas** por classe
processual, separado por ambiente.

**Como foi calculado**: filtramos `macro_desfecho == 'Concluído'`,
depois agrupamos por `ano` e `classe`.

In [137]:
# Gráfico 16 — PV, concluídos por classe
tab16 = (
    df_pv[df_pv['macro_desfecho'] == 'Concluído']
    .groupby(['ano', 'classe']).size().reset_index(name='n')
)
total16 = (
    df_pv[df_pv['macro_desfecho'] == 'Concluído']
    .groupby('ano').size().reset_index(name='n')
)

fig16 = make_subplots(specs=[[{'secondary_y': True}]])
for classe in ['ADI', 'ADPF', 'ADC', 'ADO']:
    d = tab16[tab16['classe'] == classe]
    fig16.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=classe,
        marker_color=CORES_CLASSE.get(classe, COR_TOTAL),
        text=d['n'], textposition='outside', cliponaxis=False,
    ), secondary_y=False)
fig16.add_trace(go.Scatter(
    x=total16['ano'], y=total16['n'],
    mode='lines+markers', line=dict(color=COR_LINHA, width=2),
    marker=dict(size=5), name='Total Concluídos PV',
), secondary_y=True)
fig16.update_layout(
    title_text='Gráfico 16 — Concluídos por Classe e Ano — PV',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Concluídos (por classe)'),
    yaxis2=dict(title='Total Concluídos PV'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)
fig16.show()

/tmp/ipykernel_960/26138110.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [138]:
# Gráfico 17 — PP, concluídos por classe
tab17 = (
    df_pp[df_pp['macro_desfecho'] == 'Concluído']
    .groupby(['ano', 'classe']).size().reset_index(name='n')
)
total17 = (
    df_pp[df_pp['macro_desfecho'] == 'Concluído']
    .groupby('ano').size().reset_index(name='n')
)

fig17 = make_subplots(specs=[[{'secondary_y': True}]])
for classe in ['ADI', 'ADPF', 'ADC', 'ADO']:
    d = tab17[tab17['classe'] == classe]
    if len(d) == 0:
        continue
    fig17.add_trace(go.Bar(
        x=d['ano'], y=d['n'], name=classe,
        marker_color=CORES_CLASSE.get(classe, COR_TOTAL),
        text=d['n'], textposition='outside', cliponaxis=False,
    ), secondary_y=False)
fig17.add_trace(go.Scatter(
    x=total17['ano'], y=total17['n'],
    mode='lines+markers', line=dict(color=COR_LINHA, width=2),
    marker=dict(size=5), name='Total Concluídos PP',
), secondary_y=True)
fig17.update_layout(
    title_text='Gráfico 17 — Concluídos por Classe e Ano — PP',
    template='plotly_white', barmode='group',
    margin=dict(t=80, b=120),
    xaxis=dict(dtick=1, title='Ano'),
    yaxis=dict(title='Concluídos (por classe)'),
    yaxis2=dict(title='Total Concluídos PP'),
    legend=dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5,
                font=dict(size=10), bgcolor='#fcfcfc', bordercolor='#cccccc', borderwidth=1),
)
fig17.show()

/tmp/ipykernel_960/750517755.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



### TIPO DE QUESTÃO — NÃO CONCLUÍDOS — por ano

In [139]:

# Prepara dados: não concluídos, com IJ renomeado para QI
df_nc = df_final[df_final['macro_desfecho'] == 'Não concluído'].copy()
df_nc['tipo_questao'] = df_nc['tipo_questao'].replace({'IJ': 'QI'})

# Cores por tipo de questão
CORES_TIPO = {
    'PR': '#2563eb',   # azul
    'RC': '#f59e0b',   # laranja
    'QI': '#16a34a',   # verde
}

# ---------------------------------------------------------------------------
# GRÁFICO — Plenário Virtual
# ---------------------------------------------------------------------------
df_nc_pv = df_nc[df_nc['ambiente'] == 'Plenário Virtual']
tab_pv = (
    df_nc_pv.groupby(['ano', 'tipo_questao'])
    .size()
    .reset_index(name='n')
)
total_pv = (
    df_nc_pv.groupby('ano')
    .size()
    .reset_index(name='n')
)

plotar_barras_stf(
    df_dados=tab_pv,
    col_x='ano',
    col_y='n',
    col_grupo='tipo_questao',
    titulo='Inclusões em pauta não concluídas por tipo de questão — Plenário Virtual (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_pv,
    cores=CORES_TIPO,
)

# ---------------------------------------------------------------------------
# GRÁFICO — Plenário Físico
# ---------------------------------------------------------------------------
df_nc_pp = df_nc[df_nc['ambiente'] == 'Plenário Físico']
tab_pp = (
    df_nc_pp.groupby(['ano', 'tipo_questao'])
    .size()
    .reset_index(name='n')
)
total_pp = (
    df_nc_pp.groupby('ano')
    .size()
    .reset_index(name='n')
)

plotar_barras_stf(
    df_dados=tab_pp,
    col_x='ano',
    col_y='n',
    col_grupo='tipo_questao',
    titulo='Inclusões em pauta não concluídas por tipo de questão — Plenário Físico (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_pp,
    cores=CORES_TIPO,
)


### TIPO DE QUESTÃO — CONCLUÍDOS — por ano



In [140]:

# Prepara dados: concluídos, com IJ renomeado para QI
df_c = df_final[df_final['macro_desfecho'] == 'Concluído'].copy()
df_c['tipo_questao'] = df_c['tipo_questao'].replace({'IJ': 'QI'})

CORES_TIPO = {
    'PR': '#2563eb',   # azul
    'RC': '#f59e0b',   # laranja
    'QI': '#16a34a',   # verde
}

# ---------------------------------------------------------------------------
# GRÁFICO — Plenário Virtual
# ---------------------------------------------------------------------------
df_c_pv = df_c[df_c['ambiente'] == 'Plenário Virtual']
tab_pv = (
    df_c_pv.groupby(['ano', 'tipo_questao'])
    .size()
    .reset_index(name='n')
)
total_pv = (
    df_c_pv.groupby('ano')
    .size()
    .reset_index(name='n')
)

plotar_barras_stf(
    df_dados=tab_pv,
    col_x='ano',
    col_y='n',
    col_grupo='tipo_questao',
    titulo='Inclusões em pauta concluídas por tipo de questão — Plenário Virtual (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_pv,
    cores=CORES_TIPO,
)

# ---------------------------------------------------------------------------
# GRÁFICO — Plenário Físico
# ---------------------------------------------------------------------------
df_c_pp = df_c[df_c['ambiente'] == 'Plenário Físico']
tab_pp = (
    df_c_pp.groupby(['ano', 'tipo_questao'])
    .size()
    .reset_index(name='n')
)
total_pp = (
    df_c_pp.groupby('ano')
    .size()
    .reset_index(name='n')
)

plotar_barras_stf(
    df_dados=tab_pp,
    col_x='ano',
    col_y='n',
    col_grupo='tipo_questao',
    titulo='Inclusões em pauta concluídas por tipo de questão — Plenário Físico (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_pp,
    cores=CORES_TIPO,
)


### DESFECHO CONCLUÍDO POR CATEGORIA — anual (2020-2025)

In [141]:
# ===========================================================================
# DESFECHO CONCLUÍDO POR CATEGORIA — anual (2020-2025)
# ===========================================================================

# Mapeia desfecho para as 4 categorias do gráfico
def categoria_desfecho(d):
    if d == 'Concluído - decisão unânime':
        return '1 - Unânime'
    if d == 'Concluído - decisão maioria com o relator':
        return '2 - Maioria (relator vencedor)'
    if d == 'Concluído - decisão maioria, vencido o relator':
        return '3 - Maioria (relator vencido)'
    return '4 - Não concluído (bloco)'

CORES_CATEGORIA = {
    '1 - Unânime':                    '#16a34a',  # verde
    '2 - Maioria (relator vencedor)': '#2563eb',  # azul
    '3 - Maioria (relator vencido)':  '#f59e0b',  # laranja
    '4 - Não concluído (bloco)':      '#9ca3af',  # cinza
}

df_cat = df_final.copy()
df_cat['categoria'] = df_cat['desfecho'].apply(categoria_desfecho)

# ---------------------------------------------------------------------------
# GRÁFICO — Plenário Virtual
# ---------------------------------------------------------------------------
df_cat_pv = df_cat[df_cat['ambiente'] == 'Plenário Virtual']
tab_pv = (
    df_cat_pv.groupby(['ano', 'categoria'])
    .size()
    .reset_index(name='n')
)
total_pv = (
    df_cat_pv.groupby('ano')
    .size()
    .reset_index(name='n')
)

plotar_barras_stf(
    df_dados=tab_pv,
    col_x='ano',
    col_y='n',
    col_grupo='categoria',
    titulo='Desfecho por categoria — Plenário Virtual (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_pv,
    cores=CORES_CATEGORIA,
)

# ---------------------------------------------------------------------------
# GRÁFICO — Plenário Físico
# ---------------------------------------------------------------------------
df_cat_pp = df_cat[df_cat['ambiente'] == 'Plenário Físico']
tab_pp = (
    df_cat_pp.groupby(['ano', 'categoria'])
    .size()
    .reset_index(name='n')
)
total_pp = (
    df_cat_pp.groupby('ano')
    .size()
    .reset_index(name='n')
)

_ = plotar_barras_stf(          # ← o "_ =" suprime o auto-display do Jupyter
    df_dados=tab_pp,
    col_x='ano',
    col_y='n',
    col_grupo='categoria',
    titulo='Desfecho por categoria — Plenário Físico (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_pp,
    cores=CORES_CATEGORIA,
)

### DESFECHO POR CATEGORIA — PERÍODO AGREGADO (2020-2025)

In [142]:
# ===========================================================================
# GRÁFICO 1 e 2 — DESFECHO POR CATEGORIA — PERÍODO AGREGADO (2020-2025)
# ===========================================================================

def categoria_desfecho(d):
    if d == 'Concluído - decisão unânime':
        return '1 - Unânime'
    if d == 'Concluído - decisão maioria com o relator':
        return '2 - Maioria (relator vencedor)'
    if d == 'Concluído - decisão maioria, vencido o relator':
        return '3 - Maioria (relator vencido)'
    return '4 - Não concluído (bloco)'

df_cat = df_final.copy()
df_cat['categoria'] = df_cat['desfecho'].apply(categoria_desfecho)

# --- Gráfico 1 — Plenário Virtual (período agregado) ---
serie_pv = (
    df_cat[df_cat['ambiente'] == 'Plenário Virtual']['categoria']
    .value_counts()
    .sort_index()
)
_ = plotar_pizza_stf(
    serie_pv,
    titulo='Desfecho por categoria — Plenário Virtual (2020–2025, período total)',
)

# --- Gráfico 2 — Plenário Físico (período agregado) ---
serie_pp = (
    df_cat[df_cat['ambiente'] == 'Plenário Físico']['categoria']
    .value_counts()
    .sort_index()
)
_ = plotar_pizza_stf(
    serie_pp,
    titulo='Desfecho por categoria — Plenário Físico (2020–2025, período total)',
)

### DESFECHO POR CATEGORIA e TIPO DE QUESTÃO — PERÍODO — PV

In [143]:
# ===========================================================================
#DESFECHO POR CATEGORIA e TIPO DE QUESTÃO — PERÍODO — PV
# ===========================================================================

# Renomeia IJ → QI e mantém só o Plenário Virtual
df_cat_pv = df_cat[df_cat['ambiente'] == 'Plenário Virtual'].copy()
df_cat_pv['tipo_questao'] = df_cat_pv['tipo_questao'].replace({'IJ': 'QI'})

# Cruza categoria de desfecho × tipo de questão (período agregado)
tab_cat_tipo = (
    df_cat_pv.groupby(['tipo_questao', 'categoria'])
    .size()
    .reset_index(name='n')
)

total_tipo = (
    df_cat_pv.groupby('tipo_questao')
    .size()
    .reset_index(name='n')
)

CORES_CATEGORIA = {
    '1 - Unânime':                    '#16a34a',
    '2 - Maioria (relator vencedor)': '#2563eb',
    '3 - Maioria (relator vencido)':  '#f59e0b',
    '4 - Não concluído (bloco)':      '#9ca3af',
}

_ = plotar_barras_stf(
    df_dados=tab_cat_tipo,
    col_x='tipo_questao',
    col_y='n',
    col_grupo='categoria',
    titulo='Desfecho por categoria e tipo de questão — Plenário Virtual (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_tipo,
    cores=CORES_CATEGORIA,
)

In [144]:
# ===========================================================================
# DESFECHO POR CATEGORIA e TIPO DE QUESTÃO — PERÍODO — Plenário Físico
# ===========================================================================

# Renomeia IJ → QI e mantém só o Plenário Físico
df_cat_pp = df_cat[df_cat['ambiente'] == 'Plenário Físico'].copy()
df_cat_pp['tipo_questao'] = df_cat_pp['tipo_questao'].replace({'IJ': 'QI'})

# Cruza categoria de desfecho × tipo de questão (período agregado)
tab_cat_tipo_pp = (
    df_cat_pp.groupby(['tipo_questao', 'categoria'])
    .size()
    .reset_index(name='n')
)

total_tipo_pp = (
    df_cat_pp.groupby('tipo_questao')
    .size()
    .reset_index(name='n')
)

CORES_CATEGORIA = {
    '1 - Unânime':                    '#16a34a',
    '2 - Maioria (relator vencedor)': '#2563eb',
    '3 - Maioria (relator vencido)':  '#f59e0b',
    '4 - Não concluído (bloco)':      '#9ca3af',
}

_ = plotar_barras_stf(
    df_dados=tab_cat_tipo_pp,
    col_x='tipo_questao',
    col_y='n',
    col_grupo='categoria',
    titulo='Desfecho por categoria e tipo de questão — Plenário Físico (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_tipo_pp,
    cores=CORES_CATEGORIA,
)

### DESFECHO POR CATEGORIA e TIPO DE QUESTÃO — POR ANO — Plenário Virtual e físico

In [145]:
# ===========================================================================
# DESFECHO POR CATEGORIA e TIPO DE QUESTÃO — POR ANO — Plenário Virtual
# (um gráfico por tipo de questão)
# ===========================================================================

df_cat_pv = df_cat[df_cat['ambiente'] == 'Plenário Virtual'].copy()
df_cat_pv['tipo_questao'] = df_cat_pv['tipo_questao'].replace({'IJ': 'QI'})

CORES_CATEGORIA = {
    '1 - Unânime':                    '#16a34a',
    '2 - Maioria (relator vencedor)': '#2563eb',
    '3 - Maioria (relator vencido)':  '#f59e0b',
    '4 - Não concluído (bloco)':      '#9ca3af',
}

for tipo in ['PR', 'RC', 'QI']:
    sub = df_cat_pv[df_cat_pv['tipo_questao'] == tipo]
    if sub.empty:
        continue

    tab = sub.groupby(['ano', 'categoria']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')

    _ = plotar_barras_stf(
        df_dados=tab,
        col_x='ano',
        col_y='n',
        col_grupo='categoria',
        titulo=f'Desfecho por categoria — {tipo} — Plenário Virtual (2020–2025)',
        label_y='Inclusões em pauta',
        mostrar_linha_total=True,
        df_total=total,
        cores=CORES_CATEGORIA,
    )

In [146]:
# ===========================================================================
# DESFECHO POR CATEGORIA e TIPO DE QUESTÃO — POR ANO — Plenrio Fsico
# ===========================================================================

df_cat_pp = df_cat[df_cat['ambiente'] == 'Plenrio Fsico'].copy()
df_cat_pp['tipo_questao'] = df_cat_pp['tipo_questao'].replace({'IJ': 'QI'})

CORES_CATEGORIA = {
    '1 - Unnime':                    '#16a34a',
    '2 - Maioria (relator vencedor)': '#2563eb',
    '3 - Maioria (relator vencido)':  '#f59e0b',
    '4 - No concludo (bloco)':      '#9ca3af',
}

for tipo in ['PR', 'RC', 'QI']:
    sub = df_cat_pp[df_cat_pp['tipo_questao'] == tipo]
    if sub.empty:
        continue

    tab = sub.groupby(['ano', 'categoria']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')

    # Removemos o '_' para garantir a exibio no loop
    plotar_barras_stf(
        df_dados=tab,
        col_x='ano',
        col_y='n',
        col_grupo='categoria',
        titulo=f'Desfecho por categoria — {tipo} — Plenrio Fsico (2020–2025)',
        label_y='Incluses em pauta',
        mostrar_linha_total=True,
        df_total=total,
        cores=CORES_CATEGORIA,
    )

### DESFECHO NÃO CONCLUÍDO POR CATEGORIA

In [147]:
# ===========================================================================
# DESFECHO NÃO CONCLUÍDO POR CATEGORIA — ANUAL
# ===========================================================================

# Filtra apenas os não concluídos
df_nc = df_final[df_final['macro_desfecho'] == 'Não concluído'].copy()

# Mapeia para as categorias de não concluído
def categoria_nao_concluido(d):
    if d == 'Não concluído - pedido de vista':
        return '1 - Pedido de vista'
    if d == 'Não concluído - destaque':
        return '2 - Destaque'
    if d == 'Não concluído - retirado de pauta':
        return '3 - Retirado de pauta'
    return '4 - Motivos diversos'

df_nc['categoria_nc'] = df_nc['desfecho'].apply(categoria_nao_concluido)

CORES_NC = {
    '1 - Pedido de vista':    '#8b5cf6',  # roxo
    '2 - Destaque':           '#ec4899',  # rosa
    '3 - Retirado de pauta':  '#f59e0b',  # laranja
    '4 - Motivos diversos':   '#9ca3af',  # cinza
}

# ---------------------------------------------------------------------------
# GRÁFICO — Plenário Virtual
# ---------------------------------------------------------------------------
df_nc_pv = df_nc[df_nc['ambiente'] == 'Plenário Virtual']
tab_pv = df_nc_pv.groupby(['ano', 'categoria_nc']).size().reset_index(name='n')
total_pv = df_nc_pv.groupby('ano').size().reset_index(name='n')

_ = plotar_barras_stf(
    df_dados=tab_pv,
    col_x='ano',
    col_y='n',
    col_grupo='categoria_nc',
    titulo='Desfecho não concluído por categoria — Plenário Virtual (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_pv,
    cores=CORES_NC,
)

# ---------------------------------------------------------------------------
# GRÁFICO — Plenário Físico
# ---------------------------------------------------------------------------
df_nc_pp = df_nc[df_nc['ambiente'] == 'Plenário Físico']
tab_pp = df_nc_pp.groupby(['ano', 'categoria_nc']).size().reset_index(name='n')
total_pp = df_nc_pp.groupby('ano').size().reset_index(name='n')

_ = plotar_barras_stf(
    df_dados=tab_pp,
    col_x='ano',
    col_y='n',
    col_grupo='categoria_nc',
    titulo='Desfecho não concluído por categoria — Plenário Físico (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True,
    df_total=total_pp,
    cores=CORES_NC,
)

### DESFECHO NÃO CONCLUÍDO POR CATEGORIA — ANUAL — POR CLASSE

In [148]:
# ===========================================================================
# DESFECHO NÃO CONCLUÍDO POR CATEGORIA — ANUAL — POR CLASSE
# (um gráfico por classe)
# ===========================================================================

# Filtra não concluídos e mapeia categorias
df_nc = df_final[df_final['macro_desfecho'] == 'Não concluído'].copy()

def categoria_nao_concluido(d):
    if d == 'Não concluído - pedido de vista':
        return '1 - Pedido de vista'
    if d == 'Não concluído - destaque':
        return '2 - Destaque'
    if d == 'Não concluído - retirado de pauta':
        return '3 - Retirado de pauta'
    return '4 - Motivos diversos'

df_nc['categoria_nc'] = df_nc['desfecho'].apply(categoria_nao_concluido)

CORES_NC = {
    '1 - Pedido de vista':    '#8b5cf6',
    '2 - Destaque':           '#ec4899',
    '3 - Retirado de pauta':  '#f59e0b',
    '4 - Motivos diversos':   '#9ca3af',
}

CLASSES = ['ADI', 'ADPF', 'ADC', 'ADO']

# ---------------------------------------------------------------------------
# Plenário Virtual — um gráfico por classe
# ---------------------------------------------------------------------------
df_nc_pv = df_nc[df_nc['ambiente'] == 'Plenário Virtual']

for classe in CLASSES:
    sub = df_nc_pv[df_nc_pv['classe'] == classe]
    if sub.empty:
        print(f"(PV) Sem dados de não concluído para {classe}")
        continue

    tab = sub.groupby(['ano', 'categoria_nc']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')

    _ = plotar_barras_stf(
        df_dados=tab,
        col_x='ano',
        col_y='n',
        col_grupo='categoria_nc',
        titulo=f'Não concluídos por categoria — {classe} — Plenário Virtual (2020–2025)',
        label_y='Inclusões em pauta',
        mostrar_linha_total=True,
        df_total=total,
        cores=CORES_NC,
    )

# ---------------------------------------------------------------------------
# Plenário Físico — um gráfico por classe
# ---------------------------------------------------------------------------
df_nc_pp = df_nc[df_nc['ambiente'] == 'Plenário Físico']

for classe in CLASSES:
    sub = df_nc_pp[df_nc_pp['classe'] == classe]
    if sub.empty:
        print(f"(PP) Sem dados de não concluído para {classe}")
        continue

    tab = sub.groupby(['ano', 'categoria_nc']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')

    _ = plotar_barras_stf(
        df_dados=tab,
        col_x='ano',
        col_y='n',
        col_grupo='categoria_nc',
        titulo=f'Não concluídos por categoria — {classe} — Plenário Físico (2020–2025)',
        label_y='Inclusões em pauta',
        mostrar_linha_total=True,
        df_total=total,
        cores=CORES_NC,
    )

###  DESFECHO NÃO CONCLUÍDO POR CATEGORIA — ANUAL — POR TIPO DE QUESTÃO

In [149]:
# ===========================================================================
# DESFECHO NÃO CONCLUÍDO POR CATEGORIA — ANUAL — POR TIPO DE QUESTÃO
# (um gráfico por tipo de questão)
# ===========================================================================

# Filtra não concluídos, mapeia categorias e renomeia IJ → QI
df_nc = df_final[df_final['macro_desfecho'] == 'Não concluído'].copy()
df_nc['tipo_questao'] = df_nc['tipo_questao'].replace({'IJ': 'QI'})

def categoria_nao_concluido(d):
    if d == 'Não concluído - pedido de vista':
        return '1 - Pedido de vista'
    if d == 'Não concluído - destaque':
        return '2 - Destaque'
    if d == 'Não concluído - retirado de pauta':
        return '3 - Retirado de pauta'
    return '4 - Motivos diversos'

df_nc['categoria_nc'] = df_nc['desfecho'].apply(categoria_nao_concluido)

CORES_NC = {
    '1 - Pedido de vista':    '#8b5cf6',
    '2 - Destaque':           '#ec4899',
    '3 - Retirado de pauta':  '#f59e0b',
    '4 - Motivos diversos':   '#9ca3af',
}

TIPOS = ['PR', 'RC', 'QI']

# ---------------------------------------------------------------------------
# Plenário Virtual — um gráfico por tipo de questão
# ---------------------------------------------------------------------------
df_nc_pv = df_nc[df_nc['ambiente'] == 'Plenário Virtual']

for tipo in TIPOS:
    sub = df_nc_pv[df_nc_pv['tipo_questao'] == tipo]
    if sub.empty:
        print(f"(PV) Sem dados de não concluído para {tipo}")
        continue

    tab = sub.groupby(['ano', 'categoria_nc']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')

    _ = plotar_barras_stf(
        df_dados=tab,
        col_x='ano',
        col_y='n',
        col_grupo='categoria_nc',
        titulo=f'Não concluídos por categoria — {tipo} — Plenário Virtual (2020–2025)',
        label_y='Inclusões em pauta',
        mostrar_linha_total=True,
        df_total=total,
        cores=CORES_NC,
    )

# ---------------------------------------------------------------------------
# Plenário Físico — um gráfico por tipo de questão
# ---------------------------------------------------------------------------
df_nc_pp = df_nc[df_nc['ambiente'] == 'Plenário Físico']

for tipo in TIPOS:
    sub = df_nc_pp[df_nc_pp['tipo_questao'] == tipo]
    if sub.empty:
        print(f"(PP) Sem dados de não concluído para {tipo}")
        continue

    tab = sub.groupby(['ano', 'categoria_nc']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')

    _ = plotar_barras_stf(
        df_dados=tab,
        col_x='ano',
        col_y='n',
        col_grupo='categoria_nc',
        titulo=f'Não concluídos por categoria — {tipo} — Plenário Físico (2020–2025)',
        label_y='Inclusões em pauta',
        mostrar_linha_total=True,
        df_total=total,
        cores=CORES_NC,
    )

### DESFECHO CONCLUÍDO POR CATEGORIA e TIPO DE QUESTÃO — POR ANO

In [150]:
# ===========================================================================
# DESFECHO CONCLUÍDO POR CATEGORIA e TIPO DE QUESTÃO — POR ANO
# (um gráfico por tipo de questão)
# ===========================================================================

def categoria_desfecho_ajustada(d):
    if d == 'Concluído - decisão unânime':
        return '1 - Unânime'
    if d == 'Concluído - decisão maioria com o relator':
        return '2 - Maioria (relator vencedor)'
    if d == 'Concluído - decisão maioria, vencido o relator':
        return '3 - Maioria (relator vencido)'
    return '4 - Não concluído (bloco)'

df_cat = df_final.copy()
df_cat['categoria'] = df_cat['desfecho'].apply(categoria_desfecho_ajustada)
df_cat['tipo_questao'] = df_cat['tipo_questao'].replace({'IJ': 'QI'})

CORES_CATEGORIA = {
    '1 - Unânime':                    '#16a34a',
    '2 - Maioria (relator vencedor)': '#2563eb',
    '3 - Maioria (relator vencido)':  '#f59e0b',
    '4 - Não concluído (bloco)':      '#9ca3af',
}

TIPOS = ['PR', 'RC', 'QI']

print("Gerando gráficos por tipo de questão...")

for ambiente in ['Plenário Virtual', 'Plenário Físico']:
    for tipo in TIPOS:
        sub = df_cat[(df_cat['ambiente'] == ambiente) & (df_cat['tipo_questao'] == tipo)]
        if sub.empty:
            continue

        tab = sub.groupby(['ano', 'categoria']).size().reset_index(name='n')
        total = sub.groupby('ano').size().reset_index(name='n')

        # Chamada direta sem o '_' para garantir a exibição no notebook
        plotar_barras_stf(
            df_dados=tab,
            col_x='ano',
            col_y='n',
            col_grupo='categoria',
            titulo=f'Desfecho por categoria — {tipo} — {ambiente} (2020–2025)',
            label_y='Inclusões em pauta',
            mostrar_linha_total=True,
            df_total=total,
            cores=CORES_CATEGORIA,
        )

Gerando gráficos por tipo de questão...


In [151]:
# ===========================================================================
# DESFECHO POR CATEGORIA e TIPO DE QUESTÃO — POR ANO
# ===========================================================================
# ATENÇÃO METODOLÓGICA — Plenário Físico:
# 88,3% das inclusões físicas têm tipo "Não identificado" (o complemento não
# traz sufixo de classe). Pior: os casos QUE TÊM sufixo são um resíduo de
# formato de registro, não uma amostra — PR e RC têm taxa de conclusão de
# 0,0% (0 de 174 e 0 de 63), contra 11,2% do "Não identificado", onde estão
# 268 das 276 conclusões do Físico (97%).
# Por isso a quebra por tipo de questão NÃO é gerada para o Físico: os
# gráficos sugeririam, falsamente, que o STF não conclui mérito presencial.

def categoria_desfecho_ajustada(d):
    if d == 'Concluído - decisão unânime':
        return '1 - Unânime'
    if d == 'Concluído - decisão maioria com o relator':
        return '2 - Maioria (relator vencedor)'
    if d == 'Concluído - decisão maioria, vencido o relator':
        return '3 - Maioria (relator vencido)'
    return '4 - Não concluído (bloco)'

CORES_CATEGORIA = {
    '1 - Unânime':                    '#16a34a',
    '2 - Maioria (relator vencedor)': '#2563eb',
    '3 - Maioria (relator vencido)':  '#f59e0b',
    '4 - Não concluído (bloco)':      '#9ca3af',
}

df_cat = df_final.copy()
df_cat['categoria'] = df_cat['desfecho'].apply(categoria_desfecho_ajustada)
df_cat['tipo_questao'] = df_cat['tipo_questao'].replace({'IJ': 'QI'})

# ── Plenário Virtual — quebra por tipo (dado confiável) ──────────────────
sub_pv = df_cat[df_cat['ambiente'] == 'Plenário Virtual']
print(f"{'='*58}")
print(f"PLENÁRIO VIRTUAL — {len(sub_pv):,} inclusões")
print(f"{'='*58}")

for tipo in ['PR', 'RC', 'QI']:
    sub = sub_pv[sub_pv['tipo_questao'] == tipo]
    if sub.empty:
        continue
    print(f"  {tipo:<6} {len(sub):>5,} inclusões ({100*len(sub)/len(sub_pv):>5.1f}%)")

    tab = sub.groupby(['ano', 'categoria']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')

    plotar_barras_stf(
        df_dados=tab, col_x='ano', col_y='n', col_grupo='categoria',
        titulo=f'Desfecho por categoria — {tipo} — Plenário Virtual (2020–2025)',
        label_y='Inclusões em pauta',
        mostrar_linha_total=True, df_total=total,
        cores=CORES_CATEGORIA,
    )

# ── Plenário Físico — SEM quebra por tipo ────────────────────────────────
# Um gráfico único, com todas as inclusões físicas. Mostra as 276 conclusões
# onde elas realmente estão, sem o viés do resíduo de registro.
sub_pp = df_cat[df_cat['ambiente'] == 'Plenário Físico']
print(f"\n{'='*58}")
print(f"PLENÁRIO FÍSICO — {len(sub_pp):,} inclusões (sem quebra por tipo)")
print(f"{'='*58}")
print(f"  Conclusões: {(sub_pp['macro_desfecho']=='Concluído').sum():,} "
      f"({100*(sub_pp['macro_desfecho']=='Concluído').mean():.1f}%)")
print(f"  Tipo não identificado: "
      f"{(sub_pp['tipo_questao']=='Não identificado').sum():,} (88,3%)")

tab = sub_pp.groupby(['ano', 'categoria']).size().reset_index(name='n')
total = sub_pp.groupby('ano').size().reset_index(name='n')

plotar_barras_stf(
    df_dados=tab, col_x='ano', col_y='n', col_grupo='categoria',
    titulo='Desfecho por categoria — Plenário Físico (2020–2025)',
    label_y='Inclusões em pauta',
    mostrar_linha_total=True, df_total=total,
    cores=CORES_CATEGORIA,
)

PLENÁRIO VIRTUAL — 5,422 inclusões
  PR     3,805 inclusões ( 70.2%)


  RC     1,237 inclusões ( 22.8%)


  QI       380 inclusões (  7.0%)



PLENÁRIO FÍSICO — 4,863 inclusões (sem quebra por tipo)
  Conclusões: 394 (8.1%)
  Tipo não identificado: 4,306 (88,3%)


In [152]:
# ===========================================================================
# VERIFICAÇÃO — o complemento do Físico realmente não traz sufixo?
# ===========================================================================

fis = df_pauta[~df_pauta['eh_virtual']].copy()

# 1. Quantos têm sufixo extraído
print(f"Inclusões físicas: {len(fis):,}")
print(f"  Com sufixo extraído: {fis['sufixo_extraido'].notna().sum():,}")
print(f"  Sem sufixo (None):   {fis['sufixo_extraido'].isna().sum():,}")

# 2. Como são os complementos SEM sufixo — amostra
print(f"\n{'='*62}")
print(f"AMOSTRA — complementos SEM sufixo extraído")
print(f"{'='*62}")
sem = fis[fis['sufixo_extraido'].isna()]
for t in sem['and_complemento'].dropna().head(15):
    print(f"  {str(t)[:110]!r}")

# 3. Quantos são vazios/nulos
print(f"\n  Complementos nulos:  {sem['and_complemento'].isna().sum():,}")
print(f"  Complementos vazios: {(sem['and_complemento'].astype(str).str.strip() == '').sum():,}")

# 4. Como são os complementos COM sufixo — para contraste
print(f"\n{'='*62}")
print(f"AMOSTRA — complementos COM sufixo extraído")
print(f"{'='*62}")
com = fis[fis['sufixo_extraido'].notna()]
for _, r in com.head(10).iterrows():
    print(f"  [{r['sufixo_extraido']:<12}] {str(r['and_complemento'])[:90]!r}")

# 5. Teste independente: a classe (ADI/ADPF/ADC/ADO) aparece no texto?
import re
RE_CLASSE = re.compile(r'\b(ADI|ADPF|ADC|ADO)\b', re.IGNORECASE)
tem_classe = sem['and_complemento'].astype(str).str.contains(RE_CLASSE, na=False)
print(f"\n{'='*62}")
print(f"TESTE INDEPENDENTE")
print(f"{'='*62}")
print(f"Dos {len(sem):,} sem sufixo, quantos mencionam ADI/ADPF/ADC/ADO no texto?")
print(f"  Mencionam: {tem_classe.sum():,}")
print(f"  Não mencionam: {(~tem_classe).sum():,}")
if tem_classe.sum() > 0:
    print(f"\n  Exemplos que mencionam a classe mas não tiveram sufixo extraído:")
    for t in sem[tem_classe]['and_complemento'].head(5):
        print(f"    {str(t)[:110]!r}")

# 6. Nomes de andamento mais comuns entre os sem sufixo
print(f"\n{'='*62}")
print(f"and_nome dos casos sem sufixo")
print(f"{'='*62}")
print(sem['and_nome'].value_counts().head(10).to_string())

Inclusões físicas: 6,713
  Com sufixo extraído: 787
  Sem sufixo (None):   5,926

AMOSTRA — complementos SEM sufixo extraído
  'Data de Julgamento: 16/05/2019'
  'Data de Julgamento: 15/05/2019'
  'Pleno em 09/04/2014 19:09:29'
  'Pleno Em 21/08/2009 17:39:08'
  'Pleno Em 13/02/2006 17:57:31'
  'Pleno Em 31/03/2008 17:06:36'
  'Pleno Em 13/02/2006 15:03:04'
  'Data de Julgamento: 20/04/2023'
  'Pleno em 25/02/2022 21:26:58 -'
  'Pleno Em 01/07/2008 18:03:23'
  'Data do julgamento: 1º/08/2018 (sessão das 14h)'
  'Data do julgamento: 23/05/2018'
  'Data do julgamento: 05/04/2018'
  'Data de julgamento: 27/9/2017'
  'Pleno em 20/06/2017 18:43:44'

  Complementos nulos:  83
  Complementos vazios: 0

AMOSTRA — complementos COM sufixo extraído
  [ADC-AgR     ] 'Pleno em 27/11/2015 18:56:34 - ADC-AgR'
  [ADC-MC      ] 'Pleno em 06/06/2014 16:39:21 - ADC-MC'
  [ADC-AgR     ] 'Pleno em 25/02/2015 20:07:42 - ADC-AgR'
  [ADC-AgR     ] 'Pleno em 26/04/2018 15:44:46 - ADC-AgR'
  [ADC-ED      ] 'Ple

/tmp/ipykernel_960/672616773.py:35: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



In [153]:
# ===========================================================================
# GRÁFICO 1 — DESFECHO CONCLUÍDO por CATEGORIA e TIPO DE QUESTÃO — PP
# ===========================================================================
# Ambiente: Plenário Físico | Um gráfico por tipo de questão
# 4 categorias: as 3 formas de conclusão + bloco único de não concluído

def categoria_desfecho_ajustada(d):
    if d == 'Concluído - decisão unânime':
        return '1 - Unânime'
    if d == 'Concluído - decisão maioria com o relator':
        return '2 - Maioria (relator vencedor)'
    if d == 'Concluído - decisão maioria, vencido o relator':
        return '3 - Maioria (relator vencido)'
    return '4 - Não concluído (bloco)'

CORES_CATEGORIA = {
    '1 - Unânime':                    '#16a34a',
    '2 - Maioria (relator vencedor)': '#2563eb',
    '3 - Maioria (relator vencido)':  '#f59e0b',
    '4 - Não concluído (bloco)':      '#9ca3af',
}

df_cat = df_final.copy()
df_cat['categoria'] = df_cat['desfecho'].apply(categoria_desfecho_ajustada)
df_cat['tipo_questao'] = df_cat['tipo_questao'].replace({'IJ': 'QI'})

sub_pp = df_cat[df_cat['ambiente'] == 'Plenário Físico']

# No Físico, "Não identificado" é 88,3% — precisa entrar, senão o gráfico
# representaria só 11,7% do ambiente
TIPOS_PP = ['PR', 'RC', 'QI', 'Não identificado']

print(f"{'='*58}")
print(f"PLENÁRIO FÍSICO — {len(sub_pp):,} inclusões")
print(f"{'='*58}")

for tipo in TIPOS_PP:
    sub = sub_pp[sub_pp['tipo_questao'] == tipo]
    if sub.empty:
        continue

    cobertura = 100 * len(sub) / len(sub_pp)
    print(f"  {tipo:<20} {len(sub):>5,} ({cobertura:>5.1f}% do ambiente)")

    tab = sub.groupby(['ano', 'categoria']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')

    titulo = f'Desfecho por categoria — {tipo} — Plenário Físico (2020–2025)'
    if tipo != 'Não identificado':
        titulo += f'\n[base: {len(sub)} de {len(sub_pp):,} inclusões]'

    plotar_barras_stf(
        df_dados=tab, col_x='ano', col_y='n', col_grupo='categoria',
        titulo=titulo,
        label_y='Inclusões em pauta',
        mostrar_linha_total=True, df_total=total,
        cores=CORES_CATEGORIA,
    )

PLENÁRIO FÍSICO — 4,863 inclusões
  PR                     184 (  3.8% do ambiente)


  RC                     171 (  3.5% do ambiente)


  QI                     202 (  4.2% do ambiente)


  Não identificado     4,306 ( 88.5% do ambiente)


### Processos distintos por ano e ambiente

In [154]:
# ===========================================================================
# GRÁFICO A — Processos distintos por ano e ambiente
# ===========================================================================
# Unidade: PROCESSO (incidente), não inclusão em pauta.
# Um processo pautado em 2021 e 2023 conta em ambos os anos.
# Um processo que tramitou nos dois ambientes num mesmo ano conta nas duas barras.

CORES_AMBIENTE = {
    'Plenário Virtual': '#2563eb',
    'Plenário Físico':  '#f59e0b',
}

tab = (
    df_final
    .drop_duplicates(subset=['incidente', 'ano', 'ambiente'])
    .groupby(['ano', 'ambiente']).size()
    .reset_index(name='n')
)

total = (
    df_final
    .drop_duplicates(subset=['incidente', 'ano'])
    .groupby('ano').size()
    .reset_index(name='n')
)

# Criamos o objeto da figura para ajustar o eixo Y antes de exibir
fig = plotar_barras_stf(
    df_dados=tab, col_x='ano', col_y='n', col_grupo='ambiente',
    titulo='Processos distintos por ano e ambiente (2020–2025)',
    label_y='Processos (incidentes distintos)',
    mostrar_linha_total=True, df_total=total,
    cores=CORES_AMBIENTE,
)

# Define o limite do eixo Y para 800 e exibe
fig.update_yaxes(range=[0, 800])
fig.show()

### Processos por ano e tipo de tramitação

In [155]:
# ===========================================================================
# GRÁFICO — Processos por tipo de tramitação, SEM repetição
# ===========================================================================
# Cada processo aparece UMA única vez em todo o gráfico:
#   - Ano       = ano da PRIMEIRA inclusão em pauta do processo
#   - Categoria = considera TODO o histórico do processo no período
#
# As barras somam exatamente 2.834 (total de processos distintos).

CORES_TRAMITACAO = {
    'Só Virtual':          '#2563eb',
    'Só Físico':           '#f59e0b',
    'Ambos os ambientes':  '#16a34a',
}

# Categoria de cada processo no PERÍODO INTEIRO
def classificar_tramitacao(ambientes):
    tem_v = 'Plenário Virtual' in ambientes
    tem_f = 'Plenário Físico' in ambientes
    if tem_v and tem_f:
        return 'Ambos os ambientes'
    return 'Só Virtual' if tem_v else 'Só Físico'

proc = (
    df_final.groupby('incidente')
    .agg(
        ambientes=('ambiente', set),
        ano_primeira=('data_inclusao_dt', 'min'),
    )
    .reset_index()
)
proc['tramitacao'] = proc['ambientes'].apply(classificar_tramitacao)
proc['ano'] = proc['ano_primeira'].dt.year

# Conferência
print("Processos por tipo de tramitação (período inteiro):")
print(proc['tramitacao'].value_counts().to_string())
print(f"  Total: {len(proc):,}")

tab = proc.groupby(['ano', 'tramitacao']).size().reset_index(name='n')
total = proc.groupby('ano').size().reset_index(name='n')

print(f"\nDistribuição por ano sem repetição:")
print(tab.pivot(index='ano', columns='tramitacao', values='n')
      .fillna(0).astype(int).to_string())
print(f"\nSoma de todas as barras: {tab['n'].sum():,}  (deve ser 2.834)")

plotar_barras_stf(
    df_dados=tab, col_x='ano', col_y='n', col_grupo='tramitacao',
    titulo='Processos por tipo de tramitação, por ano sem repetição',
    label_y='Processos (incidentes distintos)',
    mostrar_linha_total=True, df_total=total,
    cores=CORES_TRAMITACAO,
)

Processos por tipo de tramitação (período inteiro):
tramitacao
Só Virtual            2168
Ambos os ambientes     879
Só Físico              597
  Total: 3,644

Distribuição por ano sem repetição:
tramitacao  Ambos os ambientes  Só Físico  Só Virtual
ano                                                  
2016                       115        147          12
2017                       101         98          28
2018                       180        183          43
2019                       176         69         176
2020                        86         56         396
2021                        70         12         369
2022                        51         11         342
2023                        36         14         296
2024                        46          4         254
2025                        18          3         252

Soma de todas as barras: 3,644  (deve ser 2.834)


In [156]:
# ===========================================================================
# GRÁFICO — Processos por tipo de tramitação — PERÍODO 2020–2025
# ===========================================================================
# Sem quebra por ano. Cada processo aparece UMA ùnica vez, classificado
# pelo(s) ambiente(s) em que tramitou ao longo de todo o período.
# As barras somam exatamente 2.834 (total de processos distintos).

CORES_TRAMITACAO = {
    'Só Virtual':          '#2563eb',
    'Só Presencial':       '#f59e0b',
    'Ambos os ambientes':  '#16a34a',
}

def classificar_tramitacao(ambientes):
    tem_v = 'Plenário Virtual' in ambientes
    tem_f = 'Plenário Físico' in ambientes
    if tem_v and tem_f:
        return 'Ambos os ambientes'
    return 'Só Virtual' if tem_v else 'Só Presencial'

proc = (
    df_final.groupby('incidente')['ambiente']
    .apply(set)
    .reset_index(name='ambientes')
)
proc['tramitacao'] = proc['ambientes'].apply(classificar_tramitacao)

# Eixo X com valor ùnico — mantém a assinatura da plotar_barras_stf
proc['periodo'] = '2020–2025'

# Conferéncia
print("Processos por tipo de tramitação (2020–2025):")
print(proc['tramitacao'].value_counts().to_string())
print(f"  Total: {len(proc):,}")

tab = proc.groupby(['periodo', 'tramitacao']).size().reset_index(name='n')
total = proc.groupby('periodo').size().reset_index(name='n')

print(f"\nSoma de todas as barras: {tab['n'].sum():,}  (deve ser 2.834)")

plotar_barras_stf(
    df_dados=tab, col_x='periodo', col_y='n', col_grupo='tramitacao',
    titulo='Processos por tipo de tramitação — 2020–2025',
    label_y='Processos (incidentes distintos)',
    mostrar_linha_total=False, df_total=total,
    cores=CORES_TRAMITACAO,
)

Processos por tipo de tramitação (2020–2025):
tramitacao
Só Virtual            2168
Ambos os ambientes     879
Só Presencial          597
  Total: 3,644

Soma de todas as barras: 3,644  (deve ser 2.834)


In [157]:
print(f"PV: {df_final[df_final['ambiente']=='Plenário Virtual']['incidente'].nunique():,}")
print(f"PP: {df_final[df_final['ambiente']=='Plenário Físico']['incidente'].nunique():,}")
print(f"Total distinto: {df_final['incidente'].nunique():,}")

PV: 3,047
PP: 1,476
Total distinto: 3,644


In [158]:
# Classifica cada processo pelo(s) ambiente(s) em que tramitou
amb_por_proc = df_final.groupby('incidente')['ambiente'].apply(set)

so_virtual = (amb_por_proc == {'Plenário Virtual'}).sum()
so_fisico  = (amb_por_proc == {'Plenário Físico'}).sum()
ambos      = (amb_por_proc == {'Plenário Virtual', 'Plenário Físico'}).sum()

print(f"Só Virtual:          {so_virtual:,}")
print(f"Só Físico:           {so_fisico:,}")
print(f"Ambos os ambientes:  {ambos:,}")
print(f"{'─'*35}")
print(f"Total:               {so_virtual + so_fisico + ambos:,}")
print(f"\nConferência:")
print(f"  Só Virtual + Ambos = {so_virtual + ambos:,}  (deveria ser 2.675)")
print(f"  Só Físico + Ambos  = {so_fisico + ambos:,}  (deveria ser 637)")


Só Virtual:          2,168
Só Físico:           597
Ambos os ambientes:  879
───────────────────────────────────
Total:               3,644

Conferência:
  Só Virtual + Ambos = 3,047  (deveria ser 2.675)
  Só Físico + Ambos  = 1,476  (deveria ser 637)


### Sessoes virtuais

In [159]:
# Diagnóstico rápido antes de plotar
print("Colunas disponíveis no df_sessoes_final:")
print(df_sessoes_final.columns.tolist())

print(f"\nDistribuição de tipo_questao:")
print(df_sessoes_final['tipo_questao'].value_counts().to_string())

print(f"\nDistribuição de classe:")
print(df_sessoes_final['classe'].value_counts().to_string())

print(f"\nDistribuição de desfecho:")
print(df_sessoes_final['desfecho'].value_counts().to_string())

print(f"\nAnos disponíveis:")
print(df_sessoes_final['ano'].value_counts().sort_index().to_string())

Colunas disponíveis no df_sessoes_final:
['incidente', 'nome_processo', 'classe', 'relator', 'ano', 'data_sessao', 'data_sessao_dt', 'ambiente', 'tipo_questao', 'tipo_questao_original', 'sufixo_extraido', 'desfecho', 'macro_desfecho']

Distribuição de tipo_questao:
tipo_questao
PR    3352
RC    1128
IJ     327

Distribuição de classe:
classe
ADI     3880
ADPF     795
ADC       84
ADO       48

Distribuição de desfecho:
desfecho
Concluído - decisão unânime                       2283
Não concluído - pedido de vista                    896
Concluído - decisão maioria com o relator          823
Não concluído - retirado de pauta                  345
Não concluído - motivos diversos                   166
Não concluído - destaque                           150
Concluído - decisão maioria, vencido o relator     144

Anos disponíveis:
ano
2016     11
2017     38
2018     74
2019    349
2020    871
2021    793
2022    706
2023    827
2024    550
2025    588


In [160]:
# ===========================================================================
# GRÁFICOS — SESSÕES VIRTUAIS INICIADAS (2020–2025)
# ===========================================================================
# Unidade: cada "Iniciado Julgamento Virtual" = uma sessão.
# Base: df_sessoes_final (4.335 sessões, 2.624 processos distintos)

CORES_CLASSE = {
    'ADI': '#2563eb', 'ADPF': '#f59e0b',
    'ADC': '#16a34a', 'ADO': '#ef4444',
}
CORES_TIPO = {'PR': '#2563eb', 'RC': '#f59e0b', 'QI': '#16a34a'}
CORES_DESFECHO = {
    'Concluído - decisão unânime':                    '#16a34a',
    'Concluído - decisão maioria com o relator':      '#2563eb',
    'Concluído - decisão maioria, vencido o relator': '#f59e0b',
    'Não concluído - pedido de vista':                '#8b5cf6',
    'Não concluído - destaque':                       '#ec4899',
    'Não concluído - retirado de pauta':              '#ef4444',
    'Não concluído - motivos diversos':               '#9ca3af',
}
CORES_MACRO = {'Concluído': '#16a34a', 'Não concluído': '#9ca3af'}

df_s = df_sessoes_final.copy()
df_s['tipo_questao'] = df_s['tipo_questao'].replace({'IJ': 'QI'})
ANOS = range(2020, 2026)

# 1.1 — Sessões por ano (total)
tab = df_s.groupby('ano').size().reset_index(name='n')
tab = tab.set_index('ano').reindex(ANOS, fill_value=0).reset_index()
_ = plotar_barras_stf(tab, col_x='ano', col_y='n', titulo='Sessões virtuais iniciadas por ano (2020–2025)')

# 2.1 — Desfecho — período (pizza)
serie = df_s['desfecho'].value_counts()
_ = plotar_pizza_stf(serie, titulo='Desfecho das sessões virtuais — 2020–2025', cores=CORES_DESFECHO)

# 2.2 — Macro-desfecho — período (pizza)
serie_macro = df_s['macro_desfecho'].value_counts()
_ = plotar_pizza_stf(serie_macro, titulo='Macro-desfecho das sessões virtuais — 2020–2025', cores=CORES_MACRO)

# 2.5 — Desfecho por tipo de questão (pizza, um por tipo)
for tipo in ['PR', 'RC', 'QI']:
    sub = df_s[df_s['tipo_questao'] == tipo]
    if sub.empty: continue
    serie = sub['desfecho'].value_counts()
    _ = plotar_pizza_stf(serie, titulo=f'Desfecho — {tipo} — Sessões virtuais (2020–2025)', cores=CORES_DESFECHO)

# 2.6 — Desfecho por classe (pizza, uma por classe)
for classe in ['ADI', 'ADPF', 'ADC', 'ADO']:
    sub = df_s[df_s['classe'] == classe]
    if sub.empty: continue
    serie = sub['desfecho'].value_counts()
    _ = plotar_pizza_stf(serie, titulo=f'Desfecho — {classe} — Sessões virtuais (2020–2025)', cores=CORES_DESFECHO)

print("Gráficos corrigidos e gerados.")

Gráficos corrigidos e gerados.


In [161]:
print("Colunas do df_sessoes_final:")
print(df_sessoes_final.columns.tolist())

print(f"\nRelator — top 15:")
print(df_sessoes_final['relator'].value_counts().head(15).to_string())

print(f"\nSessões por processo (distribuição):")
sess_por_proc = df_sessoes_final.groupby('incidente').size()
print(f"  Média:   {sess_por_proc.mean():.2f}")
print(f"  Mediana: {sess_por_proc.median():.0f}")
print(f"  Máximo:  {sess_por_proc.max()}")
print(f"  1 sessão:    {(sess_por_proc == 1).sum():,} processos")
print(f"  2-5 sessões: {sess_por_proc.between(2,5).sum():,} processos")
print(f"  >5 sessões:  {(sess_por_proc > 5).sum():,} processos")

print(f"\nMês de início das sessões:")
df_sessoes_final['mes'] = df_sessoes_final['data_sessao_dt'].dt.month
print(df_sessoes_final['mes'].value_counts().sort_index().to_string())

print(f"\nCombinação classe × tipo de questão:")
print(df_sessoes_final.groupby(['classe','tipo_questao']).size().unstack(fill_value=0).to_string())

Colunas do df_sessoes_final:
['incidente', 'nome_processo', 'classe', 'relator', 'ano', 'data_sessao', 'data_sessao_dt', 'ambiente', 'tipo_questao', 'tipo_questao_original', 'sufixo_extraido', 'desfecho', 'macro_desfecho']

Relator — top 15:
relator
GILMAR MENDES           581
ALEXANDRE DE MORAES     522
LUÍS ROBERTO BARROSO    491
CÁRMEN LÚCIA            488
EDSON FACHIN            473
NUNES MARQUES           385
ROSA WEBER              380
DIAS TOFFOLI            359
LUIZ FUX                255
MARCO AURÉLIO           239
RICARDO LEWANDOWSKI     176
CRISTIANO ZANIN         127
FLÁVIO DINO             118
ANDRÉ MENDONÇA          111
CELSO DE MELLO           49

Sessões por processo (distribuição):
  Média:   1.62
  Mediana: 1
  Máximo:  15
  1 sessão:    1,865 processos
  2-5 sessões: 1,072 processos
  >5 sessões:  31 processos

Mês de início das sessões:
mes
1       1
2     461
3     413
4     377
5     397
6     579
7      12
8     637
9     517
10    541
11    490
12    382

Combin

/tmp/ipykernel_960/1000371577.py:21: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [162]:
# ===========================================================================
# GRÁFICOS — CORRELAÇÕES DAS SESSÕES VIRTUAIS
# ===========================================================================

df_s = df_sessoes_final.copy()
df_s['tipo_questao'] = df_s['tipo_questao'].replace({'IJ': 'QI'})
df_s['data_sessao_dt'] = pd.to_datetime(df_s['data_sessao_dt'])
df_s['mes'] = df_s['data_sessao_dt'].dt.month
df_s['trimestre'] = df_s['data_sessao_dt'].dt.quarter

NOMES_MESES = {
    1:'Jan', 2:'Fev', 3:'Mar', 4:'Abr', 5:'Mai', 6:'Jun',
    7:'Jul', 8:'Ago', 9:'Set', 10:'Out', 11:'Nov', 12:'Dez'
}

CORES_CLASSE = {'ADI':'#2563eb','ADPF':'#f59e0b','ADC':'#16a34a','ADO':'#ef4444'}
CORES_TIPO   = {'PR':'#2563eb','RC':'#f59e0b','QI':'#16a34a'}
CORES_MACRO  = {'Concluído':'#16a34a','Não concluído':'#9ca3af'}
CORES_DESFECHO = {
    'Concluído - decisão unânime':                    '#16a34a',
    'Concluído - decisão maioria com o relator':      '#2563eb',
    'Concluído - decisão maioria, vencido o relator': '#f59e0b',
    'Não concluído - pedido de vista':                '#8b5cf6',
    'Não concluído - destaque':                       '#ec4899',
    'Não concluído - retirado de pauta':              '#ef4444',
    'Não concluído - motivos diversos':               '#9ca3af',
}

ANOS = range(2020, 2026)


# ===========================================================================
# BLOCO 1 — SAZONALIDADE
# ===========================================================================

# 1.1 — Sessões por mês (todo o período)
tab_mes = df_s['mes'].value_counts().sort_index().reset_index()
tab_mes.columns = ['mes', 'n']
tab_mes['mes_nome'] = tab_mes['mes'].map(NOMES_MESES)
tab_mes = tab_mes.set_index('mes').reindex(range(1,13), fill_value=0).reset_index()
tab_mes['mes_nome'] = tab_mes['mes'].map(NOMES_MESES)

_ = plotar_barras_stf(
    df_dados=tab_mes.rename(columns={'mes_nome':'x'}),
    col_x='x', col_y='n', col_grupo=None,
    titulo='Sessões virtuais por mês — 2020–2025',
    label_y='Sessões',
    mostrar_linha_total=False, df_total=None,
)

# 1.2 — Sessões por mês e ano (heatmap numérico — tabela)
pivot_mes_ano = (
    df_s.groupby(['ano', 'mes']).size()
    .unstack(fill_value=0)
    .rename(columns=NOMES_MESES)
)
print("Sessões por mês e ano:")
print(pivot_mes_ano.to_string())

# 1.3 — Sessões por trimestre e ano
df_s['trim_label'] = df_s['ano'].astype(str) + ' T' + df_s['trimestre'].astype(str)
tab_trim = df_s.groupby(['ano', 'trimestre']).size().reset_index(name='n')
tab_trim['eixo'] = 'T' + tab_trim['trimestre'].astype(str)
total_trim = tab_trim.groupby('ano')['n'].sum().reset_index()

_ = plotar_barras_stf(
    df_dados=tab_trim, col_x='ano', col_y='n', col_grupo='eixo',
    titulo='Sessões virtuais por trimestre e ano (2020–2025)',
    label_y='Sessões',
    mostrar_linha_total=True, df_total=total_trim,
)


# ===========================================================================
# BLOCO 2 — RELATOR
# ===========================================================================

TOP_N = 10
top_relatores = df_s['relator'].value_counts().head(TOP_N).index.tolist()
df_top = df_s[df_s['relator'].isin(top_relatores)].copy()

# 2.1 — Sessões por relator (top 10)
tab_rel = (
    df_s['relator'].value_counts().head(TOP_N)
    .reset_index()
)
tab_rel.columns = ['relator', 'n']

_ = plotar_barras_stf(
    df_dados=tab_rel, col_x='relator', col_y='n', col_grupo=None,
    titulo='Sessões virtuais por relator — Top 10 (2020–2025)',
    label_y='Sessões',
    mostrar_linha_total=False, df_total=None,
)

# 2.2 — Taxa de conclusão por relator (top 10)
tab_taxa_rel = (
    df_top.groupby('relator')['macro_desfecho']
    .apply(lambda x: round(100 * (x == 'Concluído').mean(), 1))
    .reset_index(name='n')
    .sort_values('n', ascending=False)
)

_ = plotar_barras_stf(
    df_dados=tab_taxa_rel, col_x='relator', col_y='n', col_grupo=None,
    titulo='Taxa de conclusão por relator — Top 10 (2020–2025)',
    label_y='% de sessões concluídas',
    mostrar_linha_total=False, df_total=None,
)

# 2.3 — Desfecho por relator (top 10) — barras empilhadas
tab_desf_rel = (
    df_top.groupby(['relator', 'macro_desfecho']).size()
    .reset_index(name='n')
)

_ = plotar_barras_stf(
    df_dados=tab_desf_rel, col_x='relator', col_y='n', col_grupo='macro_desfecho',
    titulo='Macro-desfecho por relator — Top 10 (2020–2025)',
    label_y='Sessões',
    mostrar_linha_total=False, df_total=None,
    cores=CORES_MACRO,
)

# 2.4 — Classe predominante por relator (top 10)
tab_classe_rel = (
    df_top.groupby(['relator', 'classe']).size()
    .reset_index(name='n')
)

_ = plotar_barras_stf(
    df_dados=tab_classe_rel, col_x='relator', col_y='n', col_grupo='classe',
    titulo='Sessões por relator e classe — Top 10 (2020–2025)',
    label_y='Sessões',
    mostrar_linha_total=False, df_total=None,
    cores=CORES_CLASSE,
)


# ===========================================================================
# BLOCO 3 — MÚLTIPLAS SESSÕES POR PROCESSO
# ===========================================================================

sess_por_proc = df_s.groupby('incidente').size().reset_index(name='n_sessoes')
sess_por_proc = sess_por_proc.merge(
    df_s[['incidente','classe','tipo_questao','macro_desfecho']].drop_duplicates('incidente'),
    on='incidente', how='left'
)

# 3.1 — Distribuição de sessões por processo
def faixa_sessoes(n):
    if n == 1: return '1 sessão'
    if n <= 3: return '2–3 sessões'
    if n <= 5: return '4–5 sessões'
    return '6+ sessões'

sess_por_proc['faixa'] = sess_por_proc['n_sessoes'].apply(faixa_sessoes)
ORDEM_FAIXA = ['1 sessão', '2–3 sessões', '4–5 sessões', '6+ sessões']
tab_faixa = (
    sess_por_proc['faixa'].value_counts()
    .reindex(ORDEM_FAIXA, fill_value=0)
    .reset_index()
)
tab_faixa.columns = ['faixa', 'n']

_ = plotar_barras_stf(
    df_dados=tab_faixa, col_x='faixa', col_y='n', col_grupo=None,
    titulo='Distribuição de sessões por processo (2020–2025)',
    label_y='Processos',
    mostrar_linha_total=False, df_total=None,
)

# 3.2 — Faixa de sessões × classe
tab_faixa_classe = (
    sess_por_proc.groupby(['classe', 'faixa']).size()
    .reset_index(name='n')
)

_ = plotar_barras_stf(
    df_dados=tab_faixa_classe, col_x='classe', col_y='n', col_grupo='faixa',
    titulo='Número de sessões por processo e classe (2020–2025)',
    label_y='Processos',
    mostrar_linha_total=False, df_total=None,
)

# 3.3 — Taxa de conclusão na primeira sessão vs sessões posteriores
# Ordena sessões de cada processo por data
df_s_ord = df_s.sort_values(['incidente', 'data_sessao_dt']).copy()
df_s_ord['n_sessao'] = df_s_ord.groupby('incidente').cumcount() + 1
df_s_ord['eh_primeira'] = df_s_ord['n_sessao'] == 1
df_s_ord['posicao'] = df_s_ord['eh_primeira'].map({
    True: '1ª sessão', False: 'Sessões posteriores'
})

tab_pos = (
    df_s_ord.groupby('posicao')['macro_desfecho']
    .apply(lambda x: round(100 * (x == 'Concluído').mean(), 1))
    .reset_index(name='n')
)

_ = plotar_barras_stf(
    df_dados=tab_pos, col_x='posicao', col_y='n', col_grupo=None,
    titulo='Taxa de conclusão: 1ª sessão vs sessões posteriores (2020–2025)',
    label_y='% de sessões concluídas',
    mostrar_linha_total=False, df_total=None,
)

# 3.4 — Taxa de conclusão por número de sessão (1ª, 2ª, 3ª, 4ª+)
def posicao_label(n):
    if n <= 3: return f'{n}ª sessão'
    return '4ª+ sessão'

df_s_ord['posicao_n'] = df_s_ord['n_sessao'].apply(posicao_label)
ORDEM_POS = ['1ª sessão', '2ª sessão', '3ª sessão', '4ª+ sessão']

tab_pos_n = (
    df_s_ord.groupby('posicao_n')['macro_desfecho']
    .apply(lambda x: round(100 * (x == 'Concluído').mean(), 1))
    .reindex(ORDEM_POS)
    .reset_index(name='n')
)

_ = plotar_barras_stf(
    df_dados=tab_pos_n, col_x='posicao_n', col_y='n', col_grupo=None,
    titulo='Taxa de conclusão por posição da sessão no processo (2020–2025)',
    label_y='% de sessões concluídas',
    mostrar_linha_total=False, df_total=None,
)


# ===========================================================================
# BLOCO 4 — CRUZAMENTOS
# ===========================================================================

# 4.1 — Classe × tipo de questão (mapa numérico)
pivot_ct = (
    df_s.groupby(['classe', 'tipo_questao']).size()
    .unstack(fill_value=0)
)
print("\nClasse × Tipo de questão (sessões):")
print(pivot_ct.to_string())

# 4.2 — Classe × tipo de questão (barras agrupadas)
tab_ct = df_s.groupby(['classe', 'tipo_questao']).size().reset_index(name='n')
_ = plotar_barras_stf(
    df_dados=tab_ct, col_x='classe', col_y='n', col_grupo='tipo_questao',
    titulo='Sessões por classe e tipo de questão (2020–2025)',
    label_y='Sessões',
    mostrar_linha_total=False, df_total=None,
    cores=CORES_TIPO,
)

# 4.3 — Tipo de questão × desfecho por ano
for tipo in ['PR', 'RC', 'QI']:
    sub = df_s[df_s['tipo_questao'] == tipo]
    tab = sub.groupby(['ano', 'macro_desfecho']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')
    _ = plotar_barras_stf(
        df_dados=tab, col_x='ano', col_y='n', col_grupo='macro_desfecho',
        titulo=f'Macro-desfecho por ano — {tipo} — Sessões virtuais (2020–2025)',
        label_y='Sessões',
        mostrar_linha_total=True, df_total=total,
        cores=CORES_MACRO,
    )

# 4.4 — Classe × desfecho por ano
for classe in ['ADI', 'ADPF']:  # ADC e ADO com base pequena
    sub = df_s[df_s['classe'] == classe]
    tab = sub.groupby(['ano', 'macro_desfecho']).size().reset_index(name='n')
    total = sub.groupby('ano').size().reset_index(name='n')
    _ = plotar_barras_stf(
        df_dados=tab, col_x='ano', col_y='n', col_grupo='macro_desfecho',
        titulo=f'Macro-desfecho por ano — {classe} — Sessões virtuais (2020–2025)',
        label_y='Sessões',
        mostrar_linha_total=True, df_total=total,
        cores=CORES_MACRO,
    )

# 4.5 — Taxa de conclusão: classe × tipo de questão
tab_taxa_ct = (
    df_s.groupby(['classe', 'tipo_questao'])['macro_desfecho']
    .apply(lambda x: round(100 * (x == 'Concluído').mean(), 1))
    .reset_index(name='n')
)
print("\nTaxa de conclusão por classe × tipo de questão (%):")
print(tab_taxa_ct.pivot(index='classe', columns='tipo_questao', values='n')
      .fillna(0).to_string())

_ = plotar_barras_stf(
    df_dados=tab_taxa_ct, col_x='classe', col_y='n', col_grupo='tipo_questao',
    titulo='Taxa de conclusão por classe e tipo de questão — Sessões (2020–2025)',
    label_y='% de sessões concluídas',
    mostrar_linha_total=False, df_total=None,
    cores=CORES_TIPO,
)


# ===========================================================================
# BLOCO 5 — DURAÇÃO ATÉ CONCLUSÃO
# ===========================================================================
# Tempo entre a PRIMEIRA inclusão em pauta e a sessão que CONCLUIU.
# Requer df_final (inclusões) para pegar a data da primeira pauta.

df_final_virt = df_final[df_final['ambiente'] == 'Plenário Virtual'].copy()

# Primeira inclusão de cada processo
primeira_pauta = (
    df_final_virt.groupby('incidente')['data_inclusao_dt'].min()
    .reset_index(name='primeira_pauta_dt')
)

# Sessão que concluiu (a última com macro_desfecho = Concluído)
sess_concluidas = (
    df_s[df_s['macro_desfecho'] == 'Concluído']
    .sort_values('data_sessao_dt')
    .groupby('incidente').last()
    .reset_index()[['incidente', 'data_sessao_dt', 'classe', 'tipo_questao']]
)

duracao = sess_concluidas.merge(primeira_pauta, on='incidente', how='left')
duracao['dias'] = (
    duracao['data_sessao_dt'] - duracao['primeira_pauta_dt']
).dt.days
duracao = duracao[duracao['dias'] >= 0]

print(f"\nTempo (dias) entre primeira pauta e sessão de conclusão:")
print(f"  Mediana: {duracao['dias'].median():.0f} dias")
print(f"  Média:   {duracao['dias'].mean():.0f} dias")
print(f"  Mínimo:  {duracao['dias'].min()} dias")
print(f"  Máximo:  {duracao['dias'].max()} dias")
print(f"  Processos analisados: {len(duracao):,}")

def faixa_duracao(d):
    if d <= 30:   return '≤ 30 dias'
    if d <= 90:   return '31–90 dias'
    if d <= 180:  return '91–180 dias'
    if d <= 365:  return '6–12 meses'
    if d <= 730:  return '1–2 anos'
    return '> 2 anos'

ORDEM_DUR = ['≤ 30 dias','31–90 dias','91–180 dias','6–12 meses','1–2 anos','> 2 anos']
duracao['faixa_dur'] = duracao['dias'].apply(faixa_duracao)

# 5.1 — Distribuição de duração até conclusão
tab_dur = (
    duracao['faixa_dur'].value_counts()
    .reindex(ORDEM_DUR, fill_value=0)
    .reset_index()
)
tab_dur.columns = ['faixa_dur', 'n']

_ = plotar_barras_stf(
    df_dados=tab_dur, col_x='faixa_dur', col_y='n', col_grupo=None,
    titulo='Tempo até conclusão — Processos virtuais (2020–2025)',
    label_y='Processos concluídos',
    mostrar_linha_total=False, df_total=None,
)

# 5.2 — Duração mediana por classe
tab_dur_classe = (
    duracao.groupby('classe')['dias']
    .median().round(0).astype(int)
    .reset_index(name='n')
)
_ = plotar_barras_stf(
    df_dados=tab_dur_classe, col_x='classe', col_y='n', col_grupo=None,
    titulo='Tempo mediano até conclusão por classe (dias) — 2020–2025',
    label_y='Dias (mediana)',
    mostrar_linha_total=False, df_total=None,
    cores=CORES_CLASSE,
)

# 5.3 — Duração mediana por tipo de questão
tab_dur_tipo = (
    duracao.groupby('tipo_questao')['dias']
    .median().round(0).astype(int)
    .reset_index(name='n')
)
_ = plotar_barras_stf(
    df_dados=tab_dur_tipo, col_x='tipo_questao', col_y='n', col_grupo=None,
    titulo='Tempo mediano até conclusão por tipo de questão (dias) — 2020–2025',
    label_y='Dias (mediana)',
    mostrar_linha_total=False, df_total=None,
    cores=CORES_TIPO,
)

print("\nTodos os gráficos de correlação gerados.")
print(f"\nResumo por bloco:")
print(f"  Bloco 1 — Sazonalidade:           3 gráficos + 1 tabela")
print(f"  Bloco 2 — Relator:                4 gráficos")
print(f"  Bloco 3 — Múltiplas sessões:      4 gráficos")
print(f"  Bloco 4 — Cruzamentos:            5 gráficos + 2 tabelas")
print(f"  Bloco 5 — Duração até conclusão:  3 gráficos")
print(f"  Total: 19 gráficos")

Sessões por mês e ano:
mes   Jan  Fev  Mar  Abr  Mai  Jun  Jul  Ago  Set  Out  Nov  Dez
ano                                                             
2016    0    0    0    0    0    0    0    1    0    0    6    4
2017    0    2    1    0    1    6    0    4    2    8    1   13
2018    0    7   11    8    6    3    0    8    7    2   16    6
2019    0    6   12    7   12    6    0   92   69   60   51   34
2020    0   50   49  101   90  125    0  118   73  103   91   71
2021    0   69   65   71   58  106    0   78   89  137   66   54
2022    0  119   52   52   61   69   12   60   88   72   58   63
2023    1   95  101   50   58  154    0  116   79   40   80   53
2024    0   39   59   37   57   67    0   73   52   51   73   42
2025    0   74   63   51   54   43    0   87   58   68   48   42


/tmp/ipykernel_960/951067294.py:98: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/tmp/ipykernel_960/951067294.py:113: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/tmp/ipykernel_960/951067294.py:127: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/tmp/ipykernel_960/951067294.py:175: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



...


Taxa de conclusão por classe × tipo de questão (%):
tipo_questao    PR    QI    RC
classe                        
ADC           50.0  30.0  57.5
ADI           65.7  60.7  77.9
ADO           61.1   0.0  83.3
ADPF          61.5  54.8  83.8


/tmp/ipykernel_960/951067294.py:281: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




Tempo (dias) entre primeira pauta e sessão de conclusão:
  Mediana: 15 dias
  Média:   160 dias
  Mínimo:  1.0 dias
  Máximo:  2656.0 dias
  Processos analisados: 2,616


/tmp/ipykernel_960/951067294.py:361: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




Todos os gráficos de correlação gerados.

Resumo por bloco:
  Bloco 1 — Sazonalidade:           3 gráficos + 1 tabela
  Bloco 2 — Relator:                4 gráficos
  Bloco 3 — Múltiplas sessões:      4 gráficos
  Bloco 4 — Cruzamentos:            5 gráficos + 2 tabelas
  Bloco 5 — Duração até conclusão:  3 gráficos
  Total: 19 gráficos
